In [1]:
from __future__ import annotations

import os
import re
import json
import math
import time
import datetime
import random
import collections
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Literal, Union, Sequence

import numpy as np
from pydantic import BaseModel, Field, ValidationError

# -------- optional faiss --------
try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
    FAISS_IMPORT_ERROR = None
except Exception as _e:  # pragma: no cover
    faiss = None  # type: ignore
    FAISS_AVAILABLE = False
    FAISS_IMPORT_ERROR = repr(_e)

# -------- optional scipy for PXRD --------
try:
    from scipy.signal import find_peaks, peak_widths, savgol_filter  # type: ignore
except Exception:
    find_peaks = None
    peak_widths = None
    savgol_filter = None

# ========= LlamaIndex Workflows =========
from llama_index.core.workflow import Workflow, Context, step, StartEvent, StopEvent, Event
from llama_index.llms.google_genai import GoogleGenAI

# ========= DashScope embedding client (OpenAI-compatible) =========
from openai import OpenAI


# =============================================================================
# 0) CONFIG
# =============================================================================

MergeMode = Literal["deterministic", "llm_decider", "hybrid"]
PlanMode = Literal["deterministic", "llm", "llm_then_deterministic"]

@dataclass
class Settings:
    # ---- RAG v2 (set2/set3 indices) ----
    rag_db_path: str = "rag_database_v2"

    # original record file (json/jsonl). Can be absolute or relative.
    # If relative, it is resolved relative to cwd (not rag_db_path).
    records_file: str = "rag_database/cof_data.json"

    # v2 assets inside rag_db_path
    ligand_embeddings_file: str = "ligand_embeddings.npy"
    ligand_meta_file: str = "ligand_meta.json"
    ligand_index_file: str = "ligand_index.faiss"

    set2_embeddings_file: str = "set2_embeddings.npy"
    set2_meta_file: str = "set2_meta.json"
    set2_index_file: str = "set2_index.faiss"

    set3_embeddings_file: str = "set3_embeddings.npy"
    set3_meta_file: str = "set3_meta.json"
    set3_index_file: str = "set3_index.faiss"

    # ---- Embedding (DashScope/Qwen v3 embedding) ----
    dashscope_api_key_env: str = "DASHSCOPE_API_KEY"
    dashscope_base_url: str = "https://dashscope.aliyuncs.com/compatible-mode/v1"
    embedding_model: str = "text-embedding-v3"
    embedding_dim: int = 1024

    # ---- LLM (Gemini via LlamaIndex wrapper) ----
    google_api_key_env: str = "GOOGLE_API_KEY"
    llm_model: str = "gemini-2.5-pro"   #gemini-3-pro-preview
    llm_temperature: float = 0.7

    # ---- Persistence (reviewer audit) ----
    session_json_path: str = "cof_session_state.json"
    audit_jsonl_path: str = "cof_audit_log.jsonl"  # set "" to disable

    # ---- Agent1 retrieval (adaptive top-k) ----
    topk_init: int = 100
    topk_max: int = 500
    topk_grow: float = 2.0
    topk_attempts_max: int = 3

    # sufficiency thresholds (for adaptive growth; uses retrieved record evidence diversity)
    min_evid_solvent_unique: int = 6
    min_evid_catalyst_unique: int = 3

    # ---- Agent2 strategy parameters ----
    m_context: int = 50            # k: number of evidence records fed to LLM in each run
    strata: int = 5
    repeats_per_strategy: int = 1  # y: number of LLM repeats per strategy

    strategies: Tuple[str, ...] = ( "A","B","C", "D")

    # ---- Agent2 scoring (IMPORTANT) ----
    # Score uses ONLY the y priors (stability/distribution) by default.
    # Evidence stats are computed but *not* used in scoring unless you set w_evidence_ref > 0.
    w_entropy: float = 1.0
    w_jaccard: float = 1.0
    w_range_width: float = 0.15
    w_center_std: float = 0.20
    w_evidence_ref: float = 0.0   # keep 0.0 to satisfy your requirement

    # ---- Agent2 merge modes ----
    merge_mode: MergeMode = "hybrid"
    deterministic_topn_solvent: int = 3
    deterministic_topn_catalyst: int = 2
    q_low: float = 0.2
    q_mid: float = 0.5
    q_high: float = 0.8

    # ---- Agent3 experiment plan ----
    plan_mode: PlanMode = "llm_then_deterministic"
    round_experiments: int = 10
    max_rounds: int = 6

    # ---- Random seed (reproducibility) ----
    seed: int = 2025
        # ---- Prompt I/O optimization (reduce LLM anchoring noise) ----
    debug_print_prompts: bool = False

    # Agent2 prior prompt: whether to include rank/row_id/score/cof_name in evidence lines
    agent2_prior_include_record_meta: bool = False

    # Agent2 decider prompt: whether to pass computed metrics/evidence stats into LLM
    agent2_decider_pass_metrics: bool = False
    agent2_decider_pass_evidence: bool = False

    # Agent3 plan prompt: whether to pass provenance/top_neighbors into LLM
    agent3_pass_provenance: bool = False

    fewshot_enable: bool = True
    fewshot_classD_image_paths: Tuple[str, str] = (
        "pxrd/D_example.png",
        "pxrd/D_example2.png",
    )
# =============================================================================
# 1) PROMPTS (placeholders; fill with your paper prompts)
# =============================================================================

# NOTE:
# We use safe_format_prompt() instead of Python's str.format() to avoid "unescaped { }"
# problems when your prompt contains JSON examples.

PROMPT_AGENT1_EXTRACT_LIGANDS = r"""
You are an expert chemist and entity extractor. Your goal is to identify the two organic linkers (ligands) mentioned by the user for Reticular Chemistry synthesis.
- Normalize names if possible (e.g., remove "linker" or "precursor").
- If only abbreviations are given (e.g., "TAPB"), use them as is.

Return JSON only:
{
  "ligands": ["ligand1","ligand2"]
}

If user provides 3 ligands, output length=3.

User message:
{user_msg}
"""

PRIOR_CARD_SCHEMA_STR = r"""
{
  "solvent_candidates": ["...","..."],
  "catalyst_candidates": ["...","..."],
  "temperature_center": 
  "temperature_low": 
  "temperature_high": 
  "time_center_days": 
  "time_low_days": 
  "time_high_days": 
  "stoichiometry_best_string": 
  "aldehyde_per_amine_center": 
  "aldehyde_per_amine_low": 
  "aldehyde_per_amine_high":
}
"""

PROMPT_AGENT2_PRIOR = r"""
You are an expert-level researcher specializing in the methodology of Covalent Organic Framework (COF) synthesis. Your core task is to provide a SCIENTIFIC, recommendation for the experimental condition range for the synthesis of a new, unknown COF, based on a series of relevant COF synthesis cases retrieved from a database. Derive synthesis patterns from analogous cases in the database, and subsequently formulate a rational parameter space (solvent, catalyst, temperature, time) specifically tailored to this novel ligand combination.
# Target for Synthesis
This new COF will be constructed from the following ligands:
{ligands_json}
# Instructions and Framework for Analysis
Please strictly follow the steps below to meticulously analyze the provided cases and summarize the recommended condition range. 
1. Solvent System Analysis: Identify and Categorize: Identify all solvents used in the provided cases, and recommend promising solvent systems to explore.
2. Catalyst Analysis: Identify all catalysts used in the provided cases, recommend promising catalysts worthy of exploration.
3. Reaction Temperature Analysis:
Statistical Analysis: Find all specific reaction temperatures mentioned in the cases and identify the most common temperature value (mode).
Range Recommendation: Recommend a reasonable temperature range to explore. 
4. Reaction Time Analysis:
Statistical Analysis: Standardize all time units and identify the most common reaction duration.
Range Recommendation: Provide a typical reaction time range.
5. Ligand Molar Ratio Analysis:
Pattern Recognition: Analyze the molar ratios of the ligands in the provided cases. Specifically, try to find correlations between the geometry of the ligands (e.g., C3-symmetric trifunctional vs. C2-symmetric difunctional) and their molar ratios (e.g., 2:3).
Range Recommendation: Based on the inferred number of functional groups for the query ligands, recommend a theoretically sound molar ratio range.
Output Requirements:
The ultimate goal is not to provide a single optimal solution, but to offer a sufficient experimental parameter space worthy of exploration.
You MUST output JSON ONLY and MUST match this schema exactly:
{schema}

Target ligands:
{ligands_json}

Evidence records:
{evidence_block}
"""

PROMPT_AGENT2_DECIDER = r"""
You are a Senior Principal Scientist in Reticular Chemistry.
You are required to synthesize the insights from each expert to construct a comprehensive and rational Global Synthesis Prior Space. 
The ultimate goal is not to provide a single, deterministic experimental combination, but to offer a sufficient experimental parameter space worthy of exploration based on a completely new ligand combination. 
You will receive:
- selected_strategy
- metrics_summary_json (computed from y priors; stability/distribution metrics)
- evidence_summary_json (solvent/catalyst freq from selected k records; for reference only)
- prior_cards_json (y priors)

Selected strategy: {selected_strategy}
Metrics summary: {metrics_summary_json}
Evidence summary: {evidence_summary_json}
Candidate priors: {prior_cards_json}

Return JSON ONLY with this EXACT schema (no extra text):
{
  "selected_strategy": "C",
  "final_prior": {
  "solvent_candidates": ["...","..."],
  "catalyst_candidates": ["...","..."],
  "temperature_center": 
  "temperature_low": 
  "temperature_high": 
  "time_center_days": 
  "time_low_days": 
  "time_high_days": 
  "stoichiometry_best_string": 
  "aldehyde_per_amine_center": 
  "aldehyde_per_amine_low": 
  "aldehyde_per_amine_high":
    },
  "rationale": "..."
}

"""

FAILURE_TAXONOMY_TEXT = r"""
Failure taxonomy (use these exact labels):
Class A: Reaction Stalled
Phenomenon: The solution remains clear with no precipitate, or a large amount of starting material precipitates out.
Core Diagnosis: The polymerization reaction failed to initiate or proceed effectively.
Primary Causes:
A1 - Solubility Issue: At least one monomer is insoluble under the reaction conditions.
A2 - Reaction Inhibition: The catalyst concentration is too high, leading to deactivation of amine groups via over-protonation; or adverse interactions between the solvent and the catalyst/monomers.
A3 - Insufficient Reactivity: The reaction conditions (temperature, catalyst) are too mild to overcome the activation energy barrier.

Class B: Solubility Trap
Phenomenon: Formation of an oily substance, a gel, or the solution becomes viscous without any solid precipitation.
Core Diagnosis: Oligomers or polymers have formed, but their solubility is too high in the reaction solvent to allow for crystallization-driven precipitation. The solvent is a "good solvent" for the polymer network.

Class C: Kinetic Trap — Resulting in Amorphous Polymer (CMP/Amorphous COF)
Phenomenon: A powdered solid is obtained, but PXRD analysis shows it is amorphous (a broad, featureless hump). This is the most common and complex scenario.
Core Diagnosis: The reaction has occurred, but the product is locked in a disordered state. There are two opposing sub-hypotheses for this:
C1 - Runaway Reaction: The polymerization is too fast and too irreversible. Molecules collide and connect permanently in solution without sufficient time for orderly arrangement.
Inference: Requires decreasing reaction activity -> lower temperature, weaker/less catalyst, use of a modulator.
C2 - Structural Lock-in: The reaction's reversibility is severely insufficient. 
Inference: Requires enhancing the system's reversibility/dynamics -> more an effective catalyst, or changing the catalytic pathway.

Class D: Near-Crystalline / Poorly Crystalline
Phenomenon: The resulting solid shows a few broadened, low-intensity peaks in the PXRD pattern, rather than a completely amorphous hump.
Core Diagnosis: The system has found the correct path to the crystalline state, but the crystallization process is imperfect.
Primary Causes: Mismatch between nucleation and growth, insufficient reaction time, presence of minor impurities or defects.
Inference: Requires fine-tuning and optimization -> adjusting catalyst/solvent/concentration or extending reaction time, using a two-step heating profile for "annealing,"
"""

PROMPT_AGENT3_PLAN = r"""
You are an "Exploration Designer" and expert Experimental Design Chemist.
Your goal is to design an efficient and comprehensive 10-experiment initial exploration matrix (Round 1) for the synthesis of a new COF, based on the 'Final Prior' provided by a Senior Principal Scientist in Reticular Chemistry.
Generally, the current solvent system is considered viable only when the reaction yields solid powders; if a clear solution or gel-like substance is obtained, the primary objective should be the optimization of the solvent system. In cases where solid powders are produced but remain amorphous (as indicated by the absence of XRD peaks), adjustments should be made based on expert protocols, prioritized by modifying the catalyst type, acid-base concentration, or solvent combinations. Temperature and reaction time are regarded strictly as variables for optimizing crystallinity after peaks have emerged, rather than primary solutions for the absence of diffraction peaks.
So, you must generate a structured design (DoE) following these principles:
Given the hardware constraints of batch screening in a typical laboratory (single-batch isothermal conditions) and the primary objective of determining the chemical environment for nucleation, you must adhere to the following in Round 1:
Hard Constraint: LOCK the reaction temperature and time to the most recommended values from the Final Prior. Do not vary them in this round.
Variable Focus: Dedicate all 10 experiment slots to systematically screening combinations of Solvent Polarity and Catalyst Acidity, prioritizing the binary resolution of the "Nucleation vs. Non-nucleation" problem.
**LAB CONVENTIONS: Solvent Rules: The solvent scale is FIXED at 1.0 mL total volume (EXCLUDING catalyst/additives). Populate solvent_system with 
the solvent combination and solvent_detail ONLY with the ratios; the executor will convert these to volumes. 
Catalyst Rules: Catalysts are added separately (not part of the 1.0 mL total). Populate catalyst with the identity and catalyst_detail USING STANDARD
LAB SHORTHAND FOR DOSING:"X mL Y M <catalyst>(eg.0.1 mL 1.0 M HCl)"


Ligands: {ligands_json}
Final prior JSON: {final_prior_json}
Provenance: {provenance_json}
Decider rationale : {decider_rationale}

Output JSON ONLY:
{
  "round": {round_idx},
  "experiments": [
    {
      "exp_id": "R{round_idx}-E01",
      "solvent_system": "...",
      "solvent_detail": "...",
      "catalyst": "...",
      "catalyst_detail": "...",
      "temperature_C": ,
      "time_h": ,
      "stoichiometry": "",
      "notes": "..."
    }
  ]
}
"""

PROMPT_AGENT3_CLASSIFY = r"""
You are a Senior Material Scientist and PXRD Analyst.
Your task is to perform a **Holistic Diagnosis** of the experiment outcome.
Do NOT simply apply rigid thresholds. Instead, weigh the evidence from both the **Macroscopic Observations** (human text) and **Microscopic PXRD Features** (calculated metrics) to determine the "Best Fit" category.

### 1. Feature Interpretation (The "Continuum" of Quality)
Use these metrics as *indicators*, not hard switches:
- **`low_angle_peak_count_lt10`**: The hallmark of COF structure.
    - *Interpretation*: The presence of these peaks (>=1) is the strongest signal for Class D or SUCCESS. Absence usually implies Class A or C.
- **`hump_ratio_15_30`**: The "Amorphous Index".
    - *Interpretation*: Higher values indicate structural disorder (Class C). Lower values suggest a clean crystalline baseline.
- **`crystallinity_score_proxy`**: A composite metric (Peak Count × SNR / FWHM).
    - *Interpretation*: Treat this as a continuous quality score. Low (<10) is noisy/poor; High (>50) is excellent. Use it to distinguish between Class D (Poor) and SUCCESS.
- **`observations`**: The ground truth of the physical state.
    - *Interpretation*: Use this to overrule PXRD noise. If a user says "Clear Solution", the PXRD peaks are likely noise/artifacts (Class A/B).

### 2. Diagnostic Profiles (Match the data to these Archetypes)

**Class A: Reaction Stalled / Raw Material**
* *Archetype*: The user sees a clear solution or precipitates that look like starting materials.
* *PXRD Signature*: Usually flat (no peaks) OR contains sharp peaks only at high angles (>15°) corresponding to small molecule monomers. `low_angle_peak_count` is typically 0.

**Class B: Solubility Trap**
* *Archetype*: The physical state is the primary indicator: Gel, Oil, Tar, Viscous liquid.
* *PXRD Signature*: Often irrelevant because no powder exists. If measured, it looks like noise or a broad background. **Rule of thumb: If Observation says 'Gel/Oil', it is Class B, regardless of PXRD scores.**

**Class C: Kinetic Trap (Amorphous)**
* *Archetype*: A solid powder formed, but it failed to order systematically.
* *PXRD Signature*: High `hump_ratio_15_30`. Peaks are either missing (`low_angle_peak_count`=0) or extremely broad and weak. `crystallinity_score_proxy` is very low.

**Class D: Poorly Crystalline (The "Almost" COF)**
* *Archetype*: The reaction worked, structure formed, but quality is low (defects, small domains).
* *PXRD Signature*: **Crucially, it MUST have `low_angle_peak_count` >= 1.** However, the peaks might be broad (`sharp_peaks_count` low) or the SNR is poor (`crystallinity_score_proxy` is low to medium).

**SUCCESS: High Quality COF**
* *Archetype*: Highly crystalline porous solid.
* *PXRD Signature*: Distinct, sharp peaks at low angles (`low_angle_peak_count` >= 1). High `crystallinity_score_proxy`. Low `hump_ratio`.
Use this taxonomy:
{taxonomy}

Return JSON ONLY:
{
  "exp_id": "...",
  "label": "",
  "rationale": "...",
  "tactical_hints": ["...","..."]
}

Experiment package:
{exp_package_json}
"""
PROMPT_AGENT3_BATCH_ANALYZE = r"""
You are a Senior Material Scientist and PXRD Analyst.
You will receive a BATCH of experiment packages from the same round.
For EACH experiment, assign the best-fit label (Class A/B/C/D or SUCCESS) using the taxonomy.
You must weigh macroscopic observations.

Additionally, provide a summary that synthesizes patterns across experiments at "round_summary".


Use this taxonomy: {taxonomy}

Return JSON ONLY with this schema:
{
  "round_summary": "...",
  "experiments": [
    {
      "exp_id": "...",
      "label": "Class A" or "Class A"....,
      "rationale": "..."
    }
  ]
}

Batch experiment packages (JSON): {batch_json}
"""
PROMPT_AGENT3_BATCH_VISION = r"""
You are a Senior Materials Scientist specializing in X-ray Diffraction (XRD) and Covalent Organic Framework (COF) characterization.
Task: Perform a high-fidelity diagnostic analysis on a batch of synthesis experiments. You will evaluate macroscopic observations alongside PXRD plot images to classify each result according to the provided failure taxonomy
You will receive a batch of experiments. For each experiment you will get:
- exp_id
- planned condition (optional, for context)
- observations (human text)
- ONE PXRD plot image (if provided)
Failure taxonomy (use these exact labels):
{taxonomy}
PXRD Visual Analysis Protocol: When analyzing PXRD images, you must look for:
1) Artifact & background guard (especially 2°–5°):
- Treat the extreme low-angle region as high-risk for beamstop/air scatter/background.
- A real diffraction peak must be a smooth local maximum spanning multiple pixels/points with a clear rise and fall.
- Single-pixel spikes, jagged noise, or monotonic decay with small wiggles are NOT peaks.
2) Peak evidence requirement (mandatory for Class D):
- To label Class D, you MUST identify >=2 distinct broadened peaks/shoulders ABOVE the noise floor.
- In the rationale, you MUST report their approximate 2θ positions (e.g., "~3.4° and ~5.8°").
- If you cannot state >=2 peak positions, you MUST NOT assign Class D.
3) Class C criteria:
- Smooth featureless decay or a broad amorphous halo with no peaks meeting (2).
- Minor wiggles that fail the peak criteria still count as Class C (note: "possible weak ordering but not confirmed").

Few-shot Reference for Class D:  Two representative Few-shot examples of Class D (Near-Crystalline) results.
Feature Description: 
1.Distinct Multiple Bragg Peaks:The spectrum exhibits not only a dominant primary diffraction peak at low angles ($2\theta < 5^\circ$) but also at least one clearly distinguishable secondary peak or shoulder in higher-angle regions (e.g., near $5^\circ - 10^\circ$ or $20^\circ - 25^\circ$).Criterion: You MUST be able to identify at least two non-overlapping $2\theta$ positions, evidencing the presence of medium- to long-range periodic ordering.
2.Peak Morphology: Broadened but Smooth:Feature Description: Peaks display significant broadening (large Full Width at Half Maximum [FWHM]), typically corresponding to nano-scale crystallite size.
3.Significant Signal-to-Noise Ratio (SNR):Feature Description: The intensity of the diffraction peaks is significantly higher than the baseline random electronic noise or instrumental background.
4.Representative Position Examples (Reference Images):The first shows a sharp characteristic peak at $\sim 6.5^\circ$, with a distinct broad peak associated with $\pi-\pi$ stacking observed in the $\sim 22^\circ - 24^\circ$ region; the second displays a strong peak at $\sim 3.4^\circ$, accompanied by clear secondary diffraction signals at $\sim 4.5^\circ$ and $\sim 6.5^\circ$.
Criterion: The peak apex height should be at least 2–3 times the amplitude of the local background noise fluctuations, ensuring it represents a genuine structural signal rather than statistical variance.
Criterion: Although broad, each peak must consist of multiple continuous data points forming a smooth, "hill-like" profile, rather than jagged spikes composed of single pixels or random noise.
Use these as your baseline for "sufficient structural evidence." Pay close attention to the Signal-to-Noise ratio, the Full Width at Half Maximum (FWHM) of the peaks, and the requirement for multiple co-existing Bragg peaks.
Return JSON ONLY with this schema:
{{
  "round_summary": "...",
  "experiments": [
    {{
      "exp_id": "...",
      "label": "Class A|Class B|Class C|Class D",
      "rationale": "...",
    }}
  ]
}}
"""


PROMPT_AGENT3_UPDATE = r"""
You are a Strategic Diagnostician participating in a closed-loop optimization system for COF synthesis.
Experimental Context:Each round consists of 10 synthesis experiments.Feedback includes both macroscopic observations (e.g., powder formation, gels, oils) and PXRD structural data.
Each experiment is classified into one failure mode.

### Diagnostic Framework 
Analyze the classifications using this taxonomy:
1. **Class A (Reaction Stalled):** Clear solution or raw material precipitate.
   - *Diagnosis:* Kinetic barrier too high, or inhibition.
   - *Tactical Move:* Increase Acid conc, or switch to higher boiling point solvent,Increase Temperature, .
2. **Class B (Solubility Trap):** Gel, oil, or viscous solution.
   - *Diagnosis:* Solvent is too "good"; oligomers are too soluble to crystallize.
   - *Tactical Move:* Reduce the ratio of the "good solvent" (add more poor solvent) or Decrease Temperature.
3. **Class C (Amorphous Kinetic Trap):** Powder formed but amorphous (broad hump).
   - *Diagnosis C1 (Runaway):* If precipitation occurred **immediately** (< 30 mins) or yield is abnormally high (>90% quickly).. -> *Move:*Slow down  Decrease Acid, Lower Temp,or Dilute the system.
   - *Diagnosis C2 (Structural Lock-in):* If precipitation was slow/normal, but product is disordered. Reversibility insufficient. -> *Move:*Enhance Reversibility.** Increase the Catalyst Loading(Increase the added volume of the catalyst or concentration), Increase Temperature, or add a competitor modulator
4. **Class D (Near-Crystalline):** Broad peaks, low intensity.
   - *Diagnosis:* Nucleation/growth mismatch.
   - *Tactical Move:* Fine-tuning. Extend time, slight adjust in Loading, or annealing.

You will receive:
- current final prior
- a COMPACT round package: planned experiments + executed results + classifications
Your Tasks 
Propose 10 new experiments for the next round based on your updated understanding.
Analyze the feedback, identify failure modes, and diagnose based on classifications and PXRD results.
**The current solvent system is considered viable only when the reaction yields solid powders; if a clear solution or gel-like substance is obtained, the primary objective should be the optimization of the solvent system. In cases where solid powders are produced but remain amorphous, adjustments should be made based on expert protocols, prioritized by modifying the acid-base volume/concentration,catalyst type, or solvent combinations. Temperature and reaction time are regarded strictly as variables for optimizing crystallinity after peaks have emerged, rather than primary solutions for the absence of diffraction peaks.

**Priors are based on expert judgment and generally carry significant reference value; thus, exploration typically begins within the prior space. However, experimental results are the ultimate arbiter of truth. If the required adjustments lie outside the initial prior, one should not hesitate to explore beyond those defined boundaries. Should experiments conducted outside the prior prove successful, the prior must be updated accordingly to align with the experimental evidence, ensuring the system remains data-driven
Return JSON ONLY:
{
  "diagnosis_summary": "...",
  "next_round_plan": { 
  {
      "exp_id": "R{round_idx}-E01",
      "solvent_system": "...",
      "solvent_detail": "...",
      "catalyst": "...",
      "catalyst_detail": "...",
      "temperature_C": ,
      "time_h": ,
      "stoichiometry": "",
      "notes": "..."
    }
  } ...same schema as Agent3 plan...
}

Current prior: {current_prior_json}
Round history: {history_json}
**LAB CONVENTIONS:
Solvent Rules: The solvent scale is FIXED at 1.0 mL total volume (EXCLUDING catalyst/additives). Populate solvent_system with 
the solvent combination and solvent_detail ONLY with the ratios; the executor will convert these to volumes. 
Catalyst Rules: Catalysts are added separately (not part of the 1.0 mL total). Populate catalyst with the identity and catalyst_detail using standard 
lab shorthand for dosing: "X mL Y M <catalyst>"(eg.0.5 mL 1.0 M HCl)
"""


def _now_iso() -> str:
    return datetime.datetime.now().isoformat(timespec="seconds")

def normalize_name(s: str) -> str:
    """Aggressive normalization for matching ligand/solvent/catalyst names."""
    if s is None:
        return ""
    s = s.lower().strip()
    s = re.sub(r"\s+", "", s)
    s = re.sub(r"[\(\)\[\]\{\},;:'\"/\\\-–—_]+", "", s)
    return s

def safe_format_prompt(template: str, **kwargs: Any) -> str:
    """
    Safe formatter that only replaces {identifier} placeholders.
    It does NOT treat arbitrary JSON braces as format markers.
    """
    def repl(match: re.Match) -> str:
        key = match.group(1)
        if key in kwargs:
            return str(kwargs[key])
        return match.group(0)  # keep as-is
    return re.sub(r"\{([a-zA-Z_][a-zA-Z0-9_]*)\}", repl, template)

def extract_json_object(text: str) -> str:
    """
    Best-effort extraction of JSON object/array from model text.
    Handles code fences and extra text.
    """
    if text is None:
        raise ValueError("Empty text for JSON extraction.")
    text = text.strip()
    text = re.sub(r"^```(json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    first_curly = text.find("{")
    first_brack = text.find("[")
    if first_curly == -1 and first_brack == -1:
        raise ValueError(f"No JSON start found in: {text[:200]}")

    if first_curly != -1 and (first_brack == -1 or first_curly < first_brack):
        start = first_curly
        end = text.rfind("}")
    else:
        start = first_brack
        end = text.rfind("]")

    if end <= start:
        raise ValueError(f"Invalid JSON boundaries in: {text[:200]}")
    return text[start:end + 1]

async def llm_complete_text(llm: GoogleGenAI, prompt: str) -> str:
    resp = await llm.acomplete(prompt)
    return getattr(resp, "text", str(resp))

async def llm_complete_json(llm: GoogleGenAI, prompt: str, max_retries: int = 3) -> Any:
    import asyncio
    last = None
    for _ in range(max_retries):
        try:
            txt = await llm_complete_text(llm, prompt)
            js = extract_json_object(txt)
            return json.loads(js)
        except Exception as e:
            last = e
            await asyncio.sleep(0.8)
    raise RuntimeError(f"LLM JSON parse failed after retries: {last}")
from llama_index.core.llms import ChatMessage, TextBlock, ImageBlock

def guess_image_mime(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".png"]:
        return "image/png"
    if ext in [".jpg", ".jpeg"]:
        return "image/jpeg"
    if ext in [".webp"]:
        return "image/webp"
    if ext in [".heic"]:
        return "image/heic"
    if ext in [".heif"]:
        return "image/heif"
    # default
    return "image/png"

async def llm_chat_json(llm: GoogleGenAI, messages: List[ChatMessage], max_retries: int = 3) -> Any:
    import asyncio
    last = None
    for _ in range(max_retries):
        try:
            resp = await llm.achat(messages)  # async chat
            txt = getattr(resp.message, "content", None) or str(resp)
            js = extract_json_object(txt)
            return json.loads(js)
        except Exception as e:
            last = e
            await asyncio.sleep(0.8)
    raise RuntimeError(f"LLM JSON parse failed after retries: {last}")


# =============================================================================
# 3) RAG DATABASE V2 (set2/set3 indices + original records)
# =============================================================================

def _load_json_or_jsonl(path: str) -> Any:
    if path.lower().endswith(".jsonl"):
        out = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                out.append(json.loads(line))
        return out
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _resolve_path(base_dir: str, maybe_path: str) -> str:
    if os.path.isabs(maybe_path):
        return maybe_path
    if os.path.exists(maybe_path):
        return maybe_path
    return os.path.join(base_dir, maybe_path)

class RAGDatabaseV2:
    """
    Loads:
    - original COF records (json/jsonl)
    - set2/set3 indices + meta + embeddings fallback
    - ligand embeddings/meta (+ optional ligand_index)
    """
    def __init__(self, settings: Settings):
        self.s = settings
        self.records: List[Dict[str, Any]] = []

        # ligand store
        self.ligand_embeddings: Optional[np.ndarray] = None
        self.ligand_meta: List[Dict[str, Any]] = []
        self.ligand_index: Any = None

        # set2
        self.set2_embeddings: Optional[np.ndarray] = None
        self.set2_meta: List[Dict[str, Any]] = []
        self.set2_index: Any = None

        # set3
        self.set3_embeddings: Optional[np.ndarray] = None
        self.set3_meta: List[Dict[str, Any]] = []
        self.set3_index: Any = None

    def load(self) -> None:
        # records
        rec_path = self.s.records_file
        if not os.path.exists(rec_path):
            raise FileNotFoundError(f"records_file not found: {rec_path}")
        obj = _load_json_or_jsonl(rec_path)
        if not isinstance(obj, list):
            raise ValueError("records_file must be a JSON list or JSONL lines of JSON objects.")
        self.records = obj

        # v2 assets
        base = self.s.rag_db_path
        if not os.path.isdir(base):
            raise FileNotFoundError(f"rag_db_path not found: {base}")

        # load ligand store
        lig_E_path = _resolve_path(base, self.s.ligand_embeddings_file)
        lig_meta_path = _resolve_path(base, self.s.ligand_meta_file)
        if os.path.exists(lig_E_path):
            self.ligand_embeddings = np.load(lig_E_path)
        if os.path.exists(lig_meta_path):
            self.ligand_meta = _load_json_or_jsonl(lig_meta_path) or []

        lig_index_path = _resolve_path(base, self.s.ligand_index_file)
        if FAISS_AVAILABLE and os.path.exists(lig_index_path):
            try:
                self.ligand_index = faiss.read_index(lig_index_path)
            except Exception:
                self.ligand_index = None

        # load set2
        self._load_set_bucket(n=2)
        # load set3
        self._load_set_bucket(n=3)

        # sanity
        nrec = len(self.records)
        n2 = len(self.set2_meta)
        n3 = len(self.set3_meta)
        print(f"[DBv2] records={nrec} | set2={n2} | set3={n3} | faiss={FAISS_AVAILABLE}")

    def _load_set_bucket(self, n: int) -> None:
        base = self.s.rag_db_path
        if n == 2:
            emb_file, meta_file, index_file = self.s.set2_embeddings_file, self.s.set2_meta_file, self.s.set2_index_file
        elif n == 3:
            emb_file, meta_file, index_file = self.s.set3_embeddings_file, self.s.set3_meta_file, self.s.set3_index_file
        else:
            return

        emb_path = _resolve_path(base, emb_file)
        meta_path = _resolve_path(base, meta_file)
        idx_path = _resolve_path(base, index_file)

        E = np.load(emb_path) if os.path.exists(emb_path) else None
        meta = _load_json_or_jsonl(meta_path) if os.path.exists(meta_path) else []
        index = None
        if FAISS_AVAILABLE and os.path.exists(idx_path):
            try:
                index = faiss.read_index(idx_path)
            except Exception:
                index = None

        if n == 2:
            self.set2_embeddings = E
            self.set2_meta = meta or []
            self.set2_index = index
        elif n == 3:
            self.set3_embeddings = E
            self.set3_meta = meta or []
            self.set3_index = index

    def get_record(self, row_id: int) -> Dict[str, Any]:
        if 0 <= row_id < len(self.records):
            return self.records[row_id]
        return {}

    # ---------- structured helpers ----------
    @staticmethod
    def record_ligand_names(rec: Dict[str, Any]) -> List[str]:
        out = []
        for x in rec.get("ligands", []) or []:
            if isinstance(x, dict) and x.get("name"):
                out.append(str(x.get("name")))
            elif isinstance(x, str):
                out.append(x)
        return out

    @staticmethod
    def _cond_get(rec: Dict[str, Any]) -> Dict[str, Any]:
        return rec.get("synthesis_conditions", {}) or {}

    def evidence_sets_by_indices(self, record_indices: List[int]) -> Dict[str, List[str]]:
        solvents: set = set()
        catalysts: set = set()
        for i in record_indices:
            rec = self.get_record(int(i))
            cond = self._cond_get(rec)
            for s in cond.get("solvents", []) or []:
                nm = s.get("name") if isinstance(s, dict) else str(s)
                if nm:
                    solvents.add(normalize_name(nm))
            cat = cond.get("catalyst", {}) or {}
            nm = cat.get("name") if isinstance(cat, dict) else str(cat)
            if nm:
                catalysts.add(normalize_name(nm))
        # IMPORTANT: return lists (JSON-serializable)
        return {"solvents": sorted(solvents), "catalysts": sorted(catalysts)}

    def evidence_freq_by_indices(self, record_indices: List[int]) -> Dict[str, Any]:
        solvents = []
        catalysts = []
        for i in record_indices:
            rec = self.get_record(int(i))
            cond = self._cond_get(rec)
            for s in cond.get("solvents", []) or []:
                nm = s.get("name") if isinstance(s, dict) else str(s)
                if nm:
                    solvents.append(normalize_name(nm))
            cat = cond.get("catalyst", {}) or {}
            nm = cat.get("name") if isinstance(cat, dict) else str(cat)
            if nm:
                catalysts.append(normalize_name(nm))
        return {
            "solvent_freq": collections.Counter(solvents),
            "catalyst_freq": collections.Counter(catalysts),
        }

    def evidence_block_from_indices(self, record_indices: List[int], scores: Optional[List[float]] = None, m: int = 50) -> str:
        """
        Each line = compact JSON record summary (stable & low-noise for LLM).
        """
        lines = []
        m = min(m, len(record_indices))
        for r, row_id in enumerate(record_indices[:m]):
            rec = self.get_record(int(row_id))
            cof_name = rec.get("cof_name", f"ROW_{row_id}")
            ligands = self.record_ligand_names(rec)

            cond = self._cond_get(rec)
            solv = cond.get("solvents", []) or []
            solvent_names = []
            solvent_detail = []
            for s in solv:
                if isinstance(s, dict):
                    nm = s.get("name")
                    if nm:
                        solvent_names.append(nm)
                        if s.get("volume_or_ratio"):
                            solvent_detail.append(f"{nm}({s.get('volume_or_ratio')})")
                elif isinstance(s, str):
                    solvent_names.append(s)

            cat = cond.get("catalyst", {}) or {}
            cat_name = cat.get("name") if isinstance(cat, dict) else str(cat)
            cat_amt = cat.get("amount") if isinstance(cat, dict) else None

            T = (cond.get("temperature") or {}).get("value", None)
            T_unit = (cond.get("temperature") or {}).get("unit", "°C")
            tm = (cond.get("time") or {}).get("value", None)
            tm_unit = (cond.get("time") or {}).get("unit", "h")

            sc = float(scores[r]) if scores is not None and r < len(scores) and scores[r] is not None else None

            item = {
                "rank": r + 1,
                "row_id": int(row_id),
                "score": sc,  # cosine similarity (higher is better) in v2
                "cof_name": cof_name,
                "ligands": ligands,
                "solvents": solvent_names,
                "solvents_detail": solvent_detail,
                "catalyst": {"name": cat_name, "amount": cat_amt},
                "temperature": {"value": T, "unit": T_unit},
                "time": {"value": tm, "unit": tm_unit},
                "method": cond.get("method", None),
            }
            lines.append(json.dumps(item, ensure_ascii=False))
        return "\n".join(lines)
    def evidence_block_from_indices(self, record_indices: List[int], scores: Optional[List[float]] = None, m: int = 50, *, include_meta: bool = False,
        compact_json: bool = True,) -> str:
        """
        Each line = compact JSON record summary (low-noise for LLM).
        Default include_meta=False removes rank/row_id/score/cof_name to avoid anchoring.
        """
        lines = []
        m = min(m, len(record_indices))
    
        seps = (",", ":") if compact_json else None
    
        for r, row_id in enumerate(record_indices[:m]):
            rec = self.get_record(int(row_id))
            cof_name = rec.get("cof_name", f"ROW_{row_id}")
            ligands = self.record_ligand_names(rec)
    
            cond = self._cond_get(rec)
    
            solv = cond.get("solvents", []) or []
            solvent_names = []
            solvent_detail = []
            for s in solv:
                if isinstance(s, dict):
                    nm = s.get("name")
                    if nm:
                        solvent_names.append(nm)
                        if s.get("volume_or_ratio"):
                            solvent_detail.append(f"{nm}({s.get('volume_or_ratio')})")
                elif isinstance(s, str):
                    solvent_names.append(s)
    
            cat = cond.get("catalyst", {}) or {}
            cat_name = cat.get("name") if isinstance(cat, dict) else str(cat)
            cat_amt = cat.get("amount") if isinstance(cat, dict) else None
    
            T = (cond.get("temperature") or {}).get("value", None)
            T_unit = (cond.get("temperature") or {}).get("unit", "°C")
            tm = (cond.get("time") or {}).get("value", None)
            tm_unit = (cond.get("time") or {}).get("unit", "h")
    
            # ---- minimal evidence payload (what LLM truly needs) ----
            item = {
                "ligands": ligands,
                "solvents": solvent_detail if solvent_detail else solvent_names,
                "catalyst": {"name": cat_name, "amount": cat_amt},
                "temperature": {"value": T, "unit": T_unit},
                "time": {"value": tm, "unit": tm_unit},
                "method": cond.get("method", None),
            }
    
            # ---- optional meta for debugging/audit (NOT recommended for LLM) ----
            if include_meta:
                sc = float(scores[r]) if scores is not None and r < len(scores) and scores[r] is not None else None
                meta = {
                    "rank": r + 1,
                    "row_id": int(row_id),
                    "score": sc,
                    "cof_name": cof_name,
                }
                item = {**meta, **item}
    
            lines.append(json.dumps(item, ensure_ascii=False, separators=seps))
    
        return "\n".join(lines)

    def topk_summary_string(self, hits: List[Dict[str, Any]], k: int = 20) -> str:
        k = min(k, len(hits))
        rows = []
        for r in range(k):
            h = hits[r]
            row_id = int(h.get("row_id", -1))
            rec = self.get_record(row_id)
            cof_name = rec.get("cof_name", h.get("cof_name", f"ROW_{row_id}"))
            rows.append(json.dumps({
                "rank": r + 1,
                "row_id": row_id,
                "score": h.get("score"),
                "cof_name": cof_name,
                "ligands": self.record_ligand_names(rec),
            }, ensure_ascii=False))
        return "\n".join(rows)


# =============================================================================
# 4) EMBEDDER + SET RETRIEVER (2/3 ligands)
# =============================================================================

def l2norm(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    v = v.astype(np.float32, copy=False)
    n = float(np.linalg.norm(v) + eps)
    return (v / n).astype(np.float32)

def set_embedding_mean_std(vectors: Sequence[np.ndarray], eps: float = 1e-6) -> np.ndarray:
    V = np.stack([v.astype(np.float32, copy=False) for v in vectors], axis=0)
    mu = V.mean(axis=0)
    var = ((V - mu) ** 2).mean(axis=0)
    sigma = np.sqrt(var + eps).astype(np.float32)
    z = np.concatenate([mu, sigma], axis=0).astype(np.float32)
    return l2norm(z)

class DashScopeEmbedder:
    def __init__(self, settings: Settings):
        api_key = os.getenv(settings.dashscope_api_key_env, "")
        if not api_key:
            raise EnvironmentError(f"Missing env {settings.dashscope_api_key_env}")
        self.client = OpenAI(api_key=api_key, base_url=settings.dashscope_base_url)
        self.model = settings.embedding_model
        self.dim = settings.embedding_dim

    def embed(self, text: str) -> np.ndarray:
        resp = self.client.embeddings.create(model=self.model, input=text, dimensions=self.dim)
        vec = np.array(resp.model_dump()["data"][0]["embedding"], dtype=np.float32)
        return l2norm(vec)

class SetRetrieverV2:
    """
    Query using 2- or 3-ligand set embedding against set2/set3 index.
    Falls back to numpy brute-force cosine if faiss index unavailable.
    """
    def __init__(self, db: RAGDatabaseV2, embedder: DashScopeEmbedder):
        self.db = db
        self.embedder = embedder

    def search(self, ligands: List[str], top_k: int) -> Dict[str, Any]:
        ligands = [x.strip() for x in ligands if str(x).strip()]
        n = len(ligands)
        if n not in (2, 3):
            raise ValueError(f"SetRetrieverV2 only supports 2 or 3 ligands, got n={n}: {ligands}")

        # 1) embed each ligand (L2-normalized)
        vecs = [self.embedder.embed(name) for name in ligands]

        # 2) set embedding (2d, L2-normalized)
        q = set_embedding_mean_std(vecs)

        # 3) pick bucket
        if n == 2:
            index = self.db.set2_index
            E = self.db.set2_embeddings
            meta = self.db.set2_meta
            bucket = "set2"
        else:
            index = self.db.set3_index
            E = self.db.set3_embeddings
            meta = self.db.set3_meta
            bucket = "set3"

        if meta is None or len(meta) == 0:
            return {
                "bucket": bucket,
                "query_ligands": ligands,
                "indices": [],
                "scores": [],
                "hits": [],
                "records": [],
            }

        k = int(min(max(1, top_k), len(meta)))

        # 4) search
        if index is not None:
            D, I = index.search(np.array([q], dtype=np.float32), k)
            scores = D[0].tolist()
            set_ids = I[0].tolist()
        else:
            if E is None:
                raise RuntimeError(f"{bucket} faiss index unavailable and {bucket}_embeddings.npy not loaded.")
            E = np.array(E, dtype=np.float32, copy=False)
            # cosine via dot product (both L2 normalized)
            sims = E @ q.astype(np.float32)
            # top-k highest
            idx_part = np.argpartition(-sims, k - 1)[:k]
            idx_sorted = idx_part[np.argsort(-sims[idx_part])]
            set_ids = idx_sorted.tolist()
            scores = [float(sims[i]) for i in set_ids]

        # Filter -1 (faiss can return -1 if k > ntotal)
        hits: List[Dict[str, Any]] = []
        record_indices: List[int] = []
        for r, (sid, sc) in enumerate(zip(set_ids, scores), start=1):
            sid = int(sid)
            if sid < 0 or sid >= len(meta):
                continue
            m = meta[sid] or {}
            row_id = int(m.get("row_id", -1))
            if row_id < 0:
                continue
            hits.append({
                "rank": r,
                "bucket": bucket,
                "set_index_id": sid,
                "row_id": row_id,
                "score": float(sc),
                "cof_name": m.get("cof_name"),
                "ligand_names": m.get("ligand_names"),
            })
            record_indices.append(row_id)

        records = [self.db.get_record(i) for i in record_indices]

        return {
            "bucket": bucket,
            "query_ligands": ligands,
            "indices": record_indices,   # record row_id list (ranked)
            "scores": [h["score"] for h in hits],
            "hits": hits,
            "records": records,
        }


# =============================================================================
# 5) AGENT 1 — ligand extraction (2 or 3) + exact match
# =============================================================================

class LigandQuery(BaseModel):
    ligands: List[str] = Field(default_factory=list)

def heuristic_extract_ligands(user_msg: str) -> Optional[LigandQuery]:
    """
    Heuristic extractor supporting:
    - "合成 A + B"
    - "合成 A + B + C"
    - "A + B (+ C)"
    """
    if not user_msg:
        return None

    msg = user_msg.strip()
    msg = msg.replace("＋", "+")
    # remove leading verbs
    msg = re.sub(r"^\s*(我想|我要|请|帮我|麻烦|能否|希望|想)\s*", "", msg)
    msg = re.sub(r"^\s*合成\s*", "", msg)

    if "+" in msg:
        parts = [p.strip() for p in msg.split("+") if p.strip()]
        if len(parts) in (2, 3):
            return LigandQuery(ligands=parts)

    # fallback: nothing
    return None

def _coerce_ligands_from_json(js: Any) -> LigandQuery:
    """
    Backward compatible:
    - {"ligands":[...]}
    - {"ligandA":"...","ligandB":"..."} (+ optional ligandC)
    """
    if not isinstance(js, dict):
        raise ValueError("ligand extraction JSON must be an object.")
    if "ligands" in js and isinstance(js["ligands"], list):
        ligs = [str(x).strip() for x in js["ligands"] if str(x).strip()]
        return LigandQuery(ligands=ligs)
    # old style
    ligs = []
    for k in ["ligandA", "ligandB", "ligandC"]:
        if k in js and js[k]:
            ligs.append(str(js[k]).strip())
    return LigandQuery(ligands=ligs)

async def agent1_extract_ligands(llm: GoogleGenAI, user_msg: str) -> LigandQuery:
    heur = heuristic_extract_ligands(user_msg)
    if heur and len(heur.ligands) in (2, 3):
        return heur

    prompt = safe_format_prompt(PROMPT_AGENT1_EXTRACT_LIGANDS, user_msg=user_msg)
    js = await llm_complete_json(llm, prompt, max_retries=3)
    q = _coerce_ligands_from_json(js)
    if len(q.ligands) not in (2, 3):
        raise ValueError(f"Agent1 must return 2 or 3 ligands, got {q.ligands}")
    return q

def agent1_find_exact_match_multi(
    query_ligands: List[str],
    retrieved_records: List[Dict[str, Any]],
    allow_substring_fallback: bool = True,
) -> Optional[Dict[str, Any]]:
    """
    Exact match for 2/3 ligand sets:
    - strict: same count + same normalized set
    - optional: substring fallback (alias-friendly)
    """
    q_norm = [normalize_name(x) for x in query_ligands if x]
    q_set = set(q_norm)
    q_n = len(q_norm)

    for rec in retrieved_records:
        ligs = RAGDatabaseV2.record_ligand_names(rec)
        norm = [normalize_name(x) for x in ligs if x]
        if len(norm) != q_n:
            continue
        if set(norm) == q_set:
            return rec

    if allow_substring_fallback:
        # requires record contains all query ligands (normalized substring in joined string)
        for rec in retrieved_records:
            ligs = RAGDatabaseV2.record_ligand_names(rec)
            joined = " | ".join([normalize_name(x) for x in ligs])
            if all(q in joined for q in q_norm):
                # still require same count to call it "exact"
                if len(ligs) == q_n:
                    return rec
    return None

def format_direct_answer_from_record(rec: Dict[str, Any]) -> Dict[str, Any]:
    cof_name = rec.get("cof_name", "UNKNOWN")
    ligs = RAGDatabaseV2.record_ligand_names(rec)
    cond = rec.get("synthesis_conditions", {}) or {}

    solvents = cond.get("solvents", []) or []
    solvents_fmt = []
    for s in solvents:
        if isinstance(s, dict):
            solvents_fmt.append({"name": s.get("name"), "volume_or_ratio": s.get("volume_or_ratio")})
        else:
            solvents_fmt.append({"name": str(s), "volume_or_ratio": None})

    return {
        "mode": "direct",
        "cof_name": cof_name,
        "ligands": ligs,
        "synthesis_conditions": {
            "solvents": solvents_fmt,
            "temperature": cond.get("temperature"),
            "time": cond.get("time"),
            "catalyst": cond.get("catalyst"),
            "method": cond.get("method"),
        }
    }


# =============================================================================
# 6) AGENT 2 — strategy runs + scoring based on priors (NOT evidence overlap)
# =============================================================================

from pydantic import BaseModel, Field, ValidationError, ConfigDict

class PriorCard(BaseModel):
    model_config = ConfigDict(extra="allow")

    solvent_candidates: List[str] = Field(default_factory=list)
    solvent_detail: Optional[str] = None        
    catalyst_candidates: List[str] = Field(default_factory=list)
    catalyst_detail: Optional[str] = None       
    temperature_center: Optional[float] = None
    temperature_low: Optional[float] = None
    temperature_high: Optional[float] = None

    time_center_days: Optional[float] = None
    time_low_days: Optional[float] = None
    time_high_days: Optional[float] = None


    acid_low_M: Optional[float] = None
    acid_high_M: Optional[float] = None

    stoichiometry_best_string: Optional[str] = None

    aldehyde_per_amine_center: Optional[float] = None
    aldehyde_per_amine_low: Optional[float] = None
    aldehyde_per_amine_high: Optional[float] = None
    confidence: Optional[float] = None

def rank_to_strata(n: int, strata: int) -> List[int]:
    bins = np.linspace(0, n, strata + 1).astype(int)
    out = np.zeros(n, dtype=int)
    for s in range(strata):
        out[bins[s]:bins[s + 1]] = s
    return out.tolist()

def stratified_sample_indices(pool: List[int], m: int, strata: int, rng: random.Random) -> List[int]:
    """Stratified sample over rank order (pool already ranked, best first)."""
    n = len(pool)
    if n <= m:
        return list(pool)
    rank2s = rank_to_strata(n, strata)
    by = collections.defaultdict(list)
    for r, idx in enumerate(pool):
        by[rank2s[r]].append(idx)

    take = [m // strata] * strata
    for i in range(m - sum(take)):
        take[i] += 1

    picked = []
    for s in range(strata):
        candidates = by[s]
        if len(candidates) <= take[s]:
            picked.extend(candidates)
        else:
            picked.extend(rng.sample(candidates, take[s]))
    return picked

def tail_biased_stratified_sample_indices(pool: List[int], m: int, strata: int, rng: random.Random) -> List[int]:

    n = len(pool)
    if n <= m:
        return list(pool)
    rank2s = rank_to_strata(n, strata)
    by = collections.defaultdict(list)
    for r, idx in enumerate(pool):
        by[rank2s[r]].append(idx)

    # weights: 1..strata (tail gets larger weight)
    w = np.arange(1, strata + 1, dtype=float)
    w = w / w.sum()
    take = [int(round(m * float(p))) for p in w]

    # fix rounding to sum exactly m
    while sum(take) < m:
        take[-1] += 1
    while sum(take) > m:
        # remove from head first
        for s in range(strata):
            if take[s] > 0 and sum(take) > m:
                take[s] -= 1

    picked = []
    for s in range(strata):
        candidates = by.get(s, [])
        if not candidates or take[s] <= 0:
            continue
        if len(candidates) <= take[s]:
            picked.extend(candidates)
        else:
            picked.extend(rng.sample(candidates, take[s]))
    return picked

def pick_context_indices(strategy: str, ranked_indices: List[int], m: int, strata: int, seed: int) -> List[int]:
    """
    A: fixed head m
    B: head m shuffled
    C: stratified sample from full ranked list (balanced strata) + shuffle
    D: tail-biased stratified sample + shuffle
    """
    rng = random.Random(seed)
    if strategy == "A":
        return ranked_indices[:m]
    if strategy == "B":
        sel = ranked_indices[:m]
        rng.shuffle(sel)
        return sel
    if strategy == "C":
        sel = stratified_sample_indices(ranked_indices, m=m, strata=strata, rng=rng)
        rng.shuffle(sel)
        return sel
    if strategy == "D":
        sel = tail_biased_stratified_sample_indices(ranked_indices, m=m, strata=strata, rng=rng)
        rng.shuffle(sel)
        return sel
    raise ValueError(f"Unknown strategy: {strategy}")

def entropy_bits(labels: List[str]) -> float:
    if not labels:
        return 0.0
    cnt = collections.Counter(labels)
    N = sum(cnt.values())
    H = 0.0
    for v in cnt.values():
        p = v / N
        H += -p * math.log(p, 2)
    return float(H)

def pairwise_jaccard_mean(list_of_lists: List[List[str]]) -> float:
    sets = [set(map(normalize_name, x)) for x in list_of_lists if x]
    n = len(sets)
    if n < 2:
        return 1.0
    acc = 0.0
    cnt = 0
    for i in range(n):
        for j in range(i + 1, n):
            inter = len(sets[i] & sets[j])
            uni = len(sets[i] | sets[j])
            acc += (inter / uni) if uni else 1.0
            cnt += 1
    return float(acc / cnt) if cnt else 1.0

def _as_float(x: Any) -> Optional[float]:
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def _std(vals: List[Optional[float]]) -> Optional[float]:
    arr = [v for v in vals if v is not None]
    if len(arr) < 2:
        return 0.0 if arr else None
    return float(np.std(np.array(arr, dtype=float), ddof=1))

def summarize_priors_only(strategy_name: str, priors: List[PriorCard], settings: Settings) -> Dict[str, Any]:

    solvent_runs = []
    catalyst_runs = []
    solvent_flat = []
    catalyst_flat = []

    widths_T = []
    widths_time = []
    widths_acid = []
    widths_ratio = []

    # center stability
    T_center = []
    time_center = []
    acid_low = []
    acid_high = []

    for p in priors:
        sols = [normalize_name(x) for x in (p.solvent_candidates or [])][:3]
        cats = [normalize_name(x) for x in (p.catalyst_candidates or [])][:2]
        solvent_runs.append(sols)
        catalyst_runs.append(cats)
        solvent_flat.extend(sols)
        catalyst_flat.extend(cats)

        tl = _as_float(p.temperature_low); th = _as_float(p.temperature_high)
        if tl is not None and th is not None:
            widths_T.append(th - tl)
        tml = _as_float(p.time_low_days); tmh = _as_float(p.time_high_days)
        if tml is not None and tmh is not None:
            widths_time.append(tmh - tml)
        al = _as_float(p.acid_low_M); ah = _as_float(p.acid_high_M)
        if al is not None and ah is not None:
            widths_acid.append(ah - al)
        rl = _as_float(p.aldehyde_per_amine_low); rh = _as_float(p.aldehyde_per_amine_high)
        if rl is not None and rh is not None:
            widths_ratio.append(rh - rl)

        T_center.append(_as_float(p.temperature_center))
        time_center.append(_as_float(p.time_center_days))
        acid_low.append(_as_float(p.acid_low_M))
        acid_high.append(_as_float(p.acid_high_M))

    summary = {
        "name": strategy_name,
        "repeats": len(priors),

        # discrete stability
        "solvent_entropy_bits": entropy_bits(solvent_flat),
        "catalyst_entropy_bits": entropy_bits(catalyst_flat),
        "solvent_jaccard_mean": pairwise_jaccard_mean(solvent_runs),
        "catalyst_jaccard_mean": pairwise_jaccard_mean(catalyst_runs),
        "solvent_unique": len(set(solvent_flat)),
        "catalyst_unique": len(set(catalyst_flat)),

        # range widths (informativeness)
        "T_width_avg": float(np.mean(widths_T)) if widths_T else None,
        "time_width_avg": float(np.mean(widths_time)) if widths_time else None,
        "acid_width_avg": float(np.mean(widths_acid)) if widths_acid else None,
        "ratio_width_avg": float(np.mean(widths_ratio)) if widths_ratio else None,

        # center stability (std across repeats)
        "T_center_std": _std(T_center),
        "time_center_std": _std(time_center),
        "acid_low_std": _std(acid_low),
        "acid_high_std": _std(acid_high),
    }

    summary["score"] = score_strategy(summary, settings)
    return summary

def score_strategy(summary: Dict[str, Any], settings: Settings) -> float:
    """
    Lower score is better.

    Core: stability across y priors
    - entropy low
    - jaccard high
    - center std low

    Secondary: avoid absurdly wide ranges (optional penalty)
    """
    ent = float(summary.get("solvent_entropy_bits", 0.0) + summary.get("catalyst_entropy_bits", 0.0))
    jac_s = float(summary.get("solvent_jaccard_mean", 1.0))
    jac_c = float(summary.get("catalyst_jaccard_mean", 1.0))
    stability = settings.w_entropy * ent + settings.w_jaccard * ((1.0 - jac_s) + (1.0 - jac_c))

    width_pen = 0.0
    for k in ["T_width_avg", "time_width_avg", "acid_width_avg", "ratio_width_avg"]:
        w = summary.get(k, None)
        if w is None:
            continue
        width_pen += float(w)
    width_pen *= settings.w_range_width

    center_std_pen = 0.0
    for k in ["T_center_std", "time_center_std", "acid_low_std", "acid_high_std"]:
        v = summary.get(k, None)
        if v is None:
            continue
        center_std_pen += float(v)
    center_std_pen *= settings.w_center_std

    # evidence ref weight (default 0.0)
    evidence_ref_pen = 0.0
    # (kept for optional experimentation; not used by default)

    return float(stability + width_pen + center_std_pen + evidence_ref_pen)

def auto_select_strategy(strategy_summaries: Dict[str, Dict[str, Any]]) -> str:
    best = None
    best_score = float("inf")
    for name, summ in strategy_summaries.items():
        sc = float(summ.get("score", float("inf")))
        if sc < best_score:
            best = name
            best_score = sc
    return best or (list(strategy_summaries.keys())[0] if strategy_summaries else "A")

def deterministic_merge_priors(
    priors: List[PriorCard],
    topn_solvent: int,
    topn_cat: int,
    q_low: float,
    q_mid: float,
    q_high: float,
) -> Dict[str, Any]:
    """
    Deterministic merge (reproducible):
    - Discrete: pick by frequency
    - Continuous: robust quantiles
    """
    def top_counts(items: List[str], n: int) -> List[str]:
        cnt = collections.Counter([x for x in items if x])
        return [k for k, _ in cnt.most_common(n)]

    def quantile(vals: List[Optional[float]], q: float) -> Optional[float]:
        arr = [v for v in vals if v is not None]
        if not arr:
            return None
        return float(np.quantile(np.array(arr, dtype=float), q))

    sols = []
    cats = []
    T_c, T_l, T_h = [], [], []
    tm_c, tm_l, tm_h = [], [], []
    a_l, a_h = [], []
    r_c, r_l, r_h = [], [], []
    conf = []

    for p in priors:
        sols.extend(p.solvent_candidates or [])
        cats.extend(p.catalyst_candidates or [])
        T_c.append(_as_float(p.temperature_center))
        T_l.append(_as_float(p.temperature_low))
        T_h.append(_as_float(p.temperature_high))
        tm_c.append(_as_float(p.time_center_days))
        tm_l.append(_as_float(p.time_low_days))
        tm_h.append(_as_float(p.time_high_days))
        a_l.append(_as_float(p.acid_low_M))
        a_h.append(_as_float(p.acid_high_M))
        r_c.append(_as_float(p.aldehyde_per_amine_center))
        r_l.append(_as_float(p.aldehyde_per_amine_low))
        r_h.append(_as_float(p.aldehyde_per_amine_high))
        conf.append(_as_float(p.confidence))

    merged = {
        "solvent_candidates": top_counts(sols, topn_solvent),
        "catalyst_candidates": top_counts(cats, topn_cat),

        "temperature_center": quantile(T_c, q_mid),
        "temperature_low": quantile(T_l, q_low),
        "temperature_high": quantile(T_h, q_high),

        "time_center_days": quantile(tm_c, q_mid),
        "time_low_days": quantile(tm_l, q_low),
        "time_high_days": quantile(tm_h, q_high),

        "acid_low_M": quantile(a_l, q_low),
        "acid_high_M": quantile(a_h, q_high),

        "aldehyde_per_amine_center": quantile(r_c, q_mid),
        "aldehyde_per_amine_low": quantile(r_l, q_low),
        "aldehyde_per_amine_high": quantile(r_h, q_high),

        "confidence": quantile(conf, q_mid),
    }
    return merged

def _discrete_diff_metrics(evidence_items: List[str], prior_items: List[str]) -> Dict[str, Any]:
    ev = set(map(normalize_name, evidence_items or []))
    pr = set(map(normalize_name, prior_items or []))
    inter = ev & pr
    uni = ev | pr
    j = (len(inter) / len(uni)) if uni else 1.0
    novelty = (len(pr - ev) / len(pr)) if pr else 0.0
    return {
        "evidence_unique": len(ev),
        "prior_unique": len(pr),
        "jaccard": float(j),
        "prior_novelty_ratio": float(novelty),
        "prior_only": sorted(list(pr - ev))[:20],
        "evidence_only": sorted(list(ev - pr))[:20],
    }

async def agent2_run_strategy(
    llm: GoogleGenAI,
    db: RAGDatabaseV2,
    ligands: List[str],
    ranked_record_indices: List[int],
    ranked_scores: List[float],
    top_neighbors: List[Dict[str, Any]],
    strategy: str,
    settings: Settings,
) -> Tuple[List[PriorCard], Dict[str, Any]]:
    """
    One strategy:
    - select k context records (k=m_context) from Agent1 top-M retrieval
    - compute evidence stats for these k (for reference/analysis)
    - run LLM y times to get y priors
    - summarize priors-only stability and score
    """
    priors: List[PriorCard] = []
    raw_runs: List[Dict[str, Any]] = []
    valid_raw_cards: List[Dict[str, Any]] = []   # <-- 新增

    # context selection (k)
    if strategy == "E":
        ctx_ids: List[int] = []
        evidence_block = "[GLOBAL_PRIOR_CARD]\n<<<FILL if you want>>>"
    else:
        seed_base = settings.seed + (10000 if strategy == "D" else 0) + (20000 if strategy == "C" else 0) + (30000 if strategy == "B" else 0)
        ctx_ids = pick_context_indices(strategy, ranked_record_indices, m=min(settings.m_context, len(ranked_record_indices)), strata=settings.strata, seed=seed_base)

        # map record_id -> score for aligned list
        evidence_block = db.evidence_block_from_indices(
            ctx_ids,
            m=len(ctx_ids),
            include_meta=settings.agent2_prior_include_record_meta,  # default False
            compact_json=True,
        )


    # evidence stats for this strategy (m records)
    ev_sets = db.evidence_sets_by_indices(ctx_ids) if ctx_ids else {"solvents": [], "catalysts": []}
    ev_freq = db.evidence_freq_by_indices(ctx_ids) if ctx_ids else {"solvent_freq": collections.Counter(), "catalyst_freq": collections.Counter()}
    
    # run repeats
    for r in range(settings.repeats_per_strategy):
        prompt = safe_format_prompt(
            PROMPT_AGENT2_PRIOR,
            schema=PRIOR_CARD_SCHEMA_STR,
            ligands_json=json.dumps(ligands, ensure_ascii=False),
            evidence_block=evidence_block,
        )
        print("\n")
        print("prior prompt\n")
        print(prompt)
        js = await llm_complete_json(llm, prompt, max_retries=3)
        raw_runs.append(js)
        try:
            priors.append(PriorCard(**js))
            valid_raw_cards.append(js)
        except ValidationError:
            continue

    prior_summary = summarize_priors_only(strategy, priors, settings) if priors else {
        "name": strategy, "repeats": 0, "score": float("inf")
    }

    # evidence-vs-prior diffs (for your later analysis, NOT scoring)
    prior_solvents_flat = [x for p in priors for x in (p.solvent_candidates or [])]
    prior_cats_flat = [x for p in priors for x in (p.catalyst_candidates or [])]
    diff = {
        "solvents": _discrete_diff_metrics(list(ev_sets.get("solvents", [])), prior_solvents_flat),
        "catalysts": _discrete_diff_metrics(list(ev_sets.get("catalysts", [])), prior_cats_flat),
    }

    report = {
        "strategy": strategy,
        "context_k": int(len(ctx_ids)),
        "context_record_indices": ctx_ids,
        "evidence_sets": ev_sets,
        "evidence_solvent_top": ev_freq["solvent_freq"].most_common(15),
        "evidence_catalyst_top": ev_freq["catalyst_freq"].most_common(15),
        "prior_summary": prior_summary,
        "evidence_vs_prior": diff,
        "raw_runs": raw_runs,  # keep for audit
        "valid_prior_cards": valid_raw_cards,
    }
    return priors, report

async def agent2_generate_prior(
    llm: GoogleGenAI,
    db: RAGDatabaseV2,
    ligands: List[str],
    retrieval_pack: Dict[str, Any],
    settings: Settings,
) -> Dict[str, Any]:
    """
    Full Agent2 pipeline:
    - run A–E strategies
    - score strategies based on priors-only stability
    - pick best strategy
    - merge y priors into final prior:
        deterministic / llm_decider / hybrid
    """
    ranked_indices = retrieval_pack.get("indices", [])
    ranked_scores = retrieval_pack.get("scores", [])

    # provenance: top neighbors (from retrieval hits)
    neighbors = []
    for h in (retrieval_pack.get("hits") or [])[:10]:
        row_id = int(h.get("row_id", -1))
        rec = db.get_record(row_id)
        neighbors.append({
            "rank": h.get("rank"),
            "row_id": row_id,
            "score": h.get("score"),
            "cof_name": rec.get("cof_name", h.get("cof_name", f"ROW_{row_id}")),
            "ligands": db.record_ligand_names(rec),
        })

    strategy_reports: Dict[str, Any] = {}
    strategy_to_priors: Dict[str, List[PriorCard]] = {}

    for s in settings.strategies:
        priors, report = await agent2_run_strategy(
            llm=llm,
            db=db,
            ligands=ligands,
            ranked_record_indices=ranked_indices,
            ranked_scores=ranked_scores,
            top_neighbors=neighbors,
            strategy=s,
            settings=settings,
        )
        strategy_reports[s] = report
        strategy_to_priors[s] = priors

    summaries = {k: v.get("prior_summary", {}) for k, v in strategy_reports.items()}
    selected = auto_select_strategy(summaries)

    selected_priors = strategy_to_priors.get(selected, [])
    deterministic_prior = deterministic_merge_priors(
        priors=selected_priors,
        topn_solvent=settings.deterministic_topn_solvent,
        topn_cat=settings.deterministic_topn_catalyst,
        q_low=settings.q_low,
        q_mid=settings.q_mid,
        q_high=settings.q_high,
    ) if selected_priors else {}

    # --- LLM decider merge (optional) ---
    llm_decider_out: Optional[Dict[str, Any]] = None
    final_prior: Dict[str, Any] = deterministic_prior
    merge_mode_used: str = "deterministic"

    if settings.merge_mode in ("llm_decider", "hybrid"):
        # --- LLM decider merge (optional) ---
        llm_decider_out: Optional[Dict[str, Any]] = None
        final_prior: Dict[str, Any] = deterministic_prior
        merge_mode_used: str = "deterministic"
    
        if settings.merge_mode in ("llm_decider", "hybrid"):
            # 只给 decider “必要信息”
            if settings.agent2_decider_pass_metrics:
                metrics_payload = {
                    "all_strategies": summaries,
                    "selected_strategy_summary": summaries.get(selected, {}),
                }
            else:
                metrics_payload = {
                    "note": "metrics omitted to avoid anchoring; strategy already selected by script",
                    "selected_strategy": selected,
                }
        
            if settings.agent2_decider_pass_evidence:
                selected_report = strategy_reports.get(selected, {})
                evidence_payload = {
                    "top_neighbors": neighbors,
                    "selected_context_k": selected_report.get("context_k"),
                    "evidence_solvent_top": selected_report.get("evidence_solvent_top"),
                    "evidence_catalyst_top": selected_report.get("evidence_catalyst_top"),
                }
            else:
                # IMPORTANT: decider still needs target ligands
                evidence_payload = {
                    "note": "evidence omitted to avoid anchoring; base decision on prior_cards + target ligands",
                    "target_ligands": ligands,
                }
        
            selected_report = strategy_reports.get(selected, {}) or {}
            prior_cards = (
                selected_report.get("valid_prior_raw_cards")
                or selected_report.get("raw_runs")
                or []
            )
            prior_cards = prior_cards[: min(20, len(prior_cards))]

        
            decider_prompt = safe_format_prompt(
                PROMPT_AGENT2_DECIDER,
                selected_strategy=selected,
                schema=PRIOR_CARD_SCHEMA_STR,
                metrics_summary_json=json.dumps(metrics_payload, ensure_ascii=False),
                evidence_summary_json=json.dumps(evidence_payload, ensure_ascii=False),
                prior_cards_json=json.dumps(prior_cards, ensure_ascii=False),
            )
            print("\n")
            print("decider prompt\n")
            print(decider_prompt)
            # if settings.debug_print_prompts:
            #     print(decider_prompt)
            
            try:
                llm_decider_out = await llm_complete_json(llm, decider_prompt, max_retries=3)
                if isinstance(llm_decider_out, dict) and "final_prior" in llm_decider_out:
                    _ = PriorCard(**llm_decider_out["final_prior"])  # validate
                    final_prior = llm_decider_out["final_prior"]
                    merge_mode_used = "llm_decider"
                else:
                    raise ValueError("LLM decider returned no final_prior.")
            except Exception as e:
                if settings.merge_mode == "llm_decider":
                    raise
                llm_decider_out = {"error": str(e)}
                final_prior = deterministic_prior
                merge_mode_used = "deterministic_fallback"

       

    return {
        "selected_strategy": selected,
        "merge_mode_used": merge_mode_used,
        "final_prior": final_prior,
        "deterministic_prior": deterministic_prior,
        "llm_decider": llm_decider_out,
        "decider_rationale": (llm_decider_out or {}).get("rationale"),
        "strategy_reports": strategy_reports,
        "provenance": {"top_neighbors": neighbors},
    }


# =============================================================================
# 7) AGENT 3 — systematic plan + PXRD features + classification + posterior update
# =============================================================================

def _pick_first(xs: List[Any]) -> Optional[Any]:
    return xs[0] if xs else None

def deterministic_systematic_plan(prior: Dict[str, Any], round_idx: int, n_exps: int = 10) -> Dict[str, Any]:
    """
    Systematic, reproducible 10-experiment plan (NOT random).
    """
    sols = prior.get("solvent_candidates") or []
    cats = prior.get("catalyst_candidates") or []

    s1 = _pick_first(sols)
    s2 = sols[1] if len(sols) > 1 else s1
    c1 = _pick_first(cats)
    c2 = cats[1] if len(cats) > 1 else c1

    T0 = prior.get("temperature_center"); Tl = prior.get("temperature_low"); Th = prior.get("temperature_high")
    t0 = prior.get("time_center_days"); tl = prior.get("time_low_days"); th = prior.get("time_high_days")
    a0 = None
    if prior.get("acid_low_M") is not None and prior.get("acid_high_M") is not None:
        a0 = 0.5 * (prior.get("acid_low_M") + prior.get("acid_high_M"))
    al = prior.get("acid_low_M"); ah = prior.get("acid_high_M")
    ratio0 = prior.get("aldehyde_per_amine_center")
    ratioh = prior.get("aldehyde_per_amine_high")

    def days_to_h(x):
        return None if x is None else float(x) * 24.0

    base = {
        "solvent_system": s1, "solvent_detail": "",
        "catalyst": c1, "catalyst_detail": "",
        "temperature_C": T0, "time_h": days_to_h(t0),
        "acid_molarity_M": a0,
        "stoichiometry": prior.get("stoichiometry_best_string"),
        "notes": "Baseline (center) run."
    }

    plan = []
    plan.append({**base, "exp_id": f"R{round_idx}-E01"})
    plan.append({**base, "exp_id": f"R{round_idx}-E02", "temperature_C": Tl, "notes": "Low temperature boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E03", "temperature_C": Th, "notes": "High temperature boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E04", "time_h": days_to_h(tl), "notes": "Short time boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E05", "time_h": days_to_h(th), "notes": "Long time boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E06", "acid_molarity_M": al, "notes": "Low acid boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E07", "acid_molarity_M": ah, "notes": "High acid boundary."})
    plan.append({**base, "exp_id": f"R{round_idx}-E08", "solvent_system": s2, "notes": "Alternative solvent candidate."})
    plan.append({**base, "exp_id": f"R{round_idx}-E09", "catalyst": c2, "notes": "Alternative catalyst candidate."})

    e10 = {**base, "exp_id": f"R{round_idx}-E10", "temperature_C": Th, "time_h": days_to_h(th), "notes": "Combined boundary exploration."}
    if ratioh is not None:
        e10["aldehyde_per_amine"] = ratioh
        e10["notes"] += " Also high aldehyde/amine ratio."
    plan.append(e10)

    return {"round": round_idx, "experiments": plan[:n_exps], "notes": "Deterministic systematic plan (fallback)."}

def load_pxrd_two_column(path: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Robust parser for PXRD exports (.xy/.xye/.txt/.csv/.xls-as-text).
    Accept only lines where first two tokens are numeric.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"PXRD file not found: {path}")

    xs, ys = [], []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(("#", "!", "//", ";")):
                continue
            parts = re.split(r"[,\s;]+", line)
            if len(parts) < 2:
                continue
            try:
                x = float(parts[0]); y = float(parts[1])
            except Exception:
                continue
            xs.append(x); ys.append(y)

    if len(xs) < 10:
        raise ValueError(f"Too few PXRD points parsed from {path}: {len(xs)}")
    return np.array(xs, dtype=float), np.array(ys, dtype=float)


def compute_pxrd_features(two_theta: np.ndarray, intensity: np.ndarray) -> Dict[str, Any]:
    """
    PXRD feature set (SciPy preferred, robust fallback allowed).

    Key fixes:
    - y_ref is computed from 5–30° (avoids low-angle background dominating thresholds).
    - low_angle_peak_count_lt10 is made MORE ROBUST:
        * counts peaks in 2–10° only
        * requires relative prominence >= LOW_REL_PROM_MIN to suppress beamstop/background ripples
    - snr_max uses a noise floor tied to y_ref to avoid absurdly huge SNR when noise is underestimated.

    Notes:
    - We keep original feature keys (low_angle_peak_count_lt10, snr_max, crystallinity_score_proxy, hump_ratio_15_30),
      so your current classifier prompt can benefit without modification.
    """
    x = np.array(two_theta, dtype=float).copy()
    y = np.array(intensity, dtype=float).copy()

    order = np.argsort(x)
    x = x[order]
    y = y[order]
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    dx = float(np.median(np.diff(x))) if len(x) > 1 else 1.0

    # ---- smoothing ----
    y_s = y
    if savgol_filter is not None and len(y) >= 31:
        win = min(101, (len(y) // 2) * 2 + 1)
        win = max(win, 31)
        try:
            y_s = savgol_filter(y, window_length=win, polyorder=3)
        except Exception:
            y_s = y
    else:
        # fallback smoothing: moving average ~0.8°
        win_pts = int(max(31, min(201, round(0.8 / max(dx, 1e-9)))))
        if win_pts % 2 == 0:
            win_pts += 1
        pad = win_pts // 2
        ypad = np.pad(y, (pad, pad), mode="reflect")
        kernel = np.ones(win_pts, dtype=float) / win_pts
        y_s = np.convolve(ypad, kernel, mode="valid")

    # ---- baseline & residual ----
    baseline = float(np.percentile(y_s, 10))
    resid = y_s - baseline

    # ---- noise estimate (peak detection) ----
    # Use derivative MAD on 10–30° to reduce low-angle slope artifacts.
    noise_mask = (x >= 10.0) & (x <= 30.0)
    dy = np.diff(resid[noise_mask]) if np.count_nonzero(noise_mask) >= 10 else np.diff(resid)
    mad = float(np.median(np.abs(dy - np.median(dy))) + 1e-12)
    noise_peak = 1.4826 * mad

    # ---- reference intensity (avoid <5° low-angle background) ----
    ref_mask = (x >= 5.0) & (x <= 30.0)
    y_ref = float(np.percentile(y_s[ref_mask], 99)) if np.count_nonzero(ref_mask) >= 10 else float(np.percentile(y_s, 99))

    # ---- peak detection ----
    peaks = np.array([], dtype=int)
    prominences = np.array([], dtype=float)
    fwhm = []

    if find_peaks is not None:
        height = baseline + max(3.0 * noise_peak, 0.01 * y_ref)
        prom_thr = max(2.0 * noise_peak, 0.005 * y_ref)
        try:
            peaks, props = find_peaks(
                y_s,
                height=height,
                prominence=prom_thr,
                distance=max(3, int(round(0.25 / max(dx, 1e-9)))),
            )
            prominences = props.get("prominences", np.array([], dtype=float))
            if not isinstance(prominences, np.ndarray):
                prominences = np.array(list(prominences), dtype=float) if prominences is not None else np.array([], dtype=float)
        except Exception:
            peaks = np.array([], dtype=int)
            prominences = np.array([], dtype=float)

        if peak_widths is not None and len(peaks):
            try:
                widths_res = peak_widths(y_s, peaks, rel_height=0.5)
                fwhm = (widths_res[0] * dx).tolist()
            except Exception:
                fwhm = []
    else:
        # ---- robust fallback (no SciPy) ----
        if len(y_s) >= 3:
            cand = np.where((y_s[1:-1] > y_s[:-2]) & (y_s[1:-1] > y_s[2:]))[0] + 1
        else:
            cand = np.array([], dtype=int)

        height_thr = baseline + max(3.0 * noise_peak, 0.01 * y_ref)
        prom_thr = max(2.0 * noise_peak, 0.005 * y_ref)

        if cand.size:
            cand = cand[y_s[cand] >= height_thr]

        # prominence in a local window (~0.8°)
        w = max(5, int(round(0.8 / max(dx, 1e-9))))
        if cand.size:
            prom = []
            for i in cand:
                l0 = max(0, i - w)
                r0 = min(len(y_s), i + w + 1)
                left = y_s[l0:i]
                right = y_s[i + 1:r0]
                if left.size == 0 or right.size == 0:
                    prom.append(0.0)
                else:
                    prom.append(float(y_s[i] - max(left.min(), right.min())))
            prom = np.array(prom, dtype=float)
            keep = prom >= prom_thr
            cand = cand[keep]
            prom = prom[keep]
        else:
            prom = np.array([], dtype=float)

        # min-distance filtering (~0.25°): keep highest peaks
        dist = max(3, int(round(0.25 / max(dx, 1e-9))))
        if cand.size:
            order2 = np.argsort(-y_s[cand])
            chosen = []
            chosen_prom = []
            for idx in order2:
                p = int(cand[idx])
                if all(abs(p - c) >= dist for c in chosen):
                    chosen.append(p)
                    chosen_prom.append(float(prom[idx]) if prom.size else 0.0)
            sort_idx = np.argsort(chosen)
            peaks = np.array([chosen[i] for i in sort_idx], dtype=int)
            prominences = np.array([chosen_prom[i] for i in sort_idx], dtype=float)

        # simple FWHM estimate
        if len(peaks):
            for i in peaks:
                y_peak = float(y_s[i])
                half = baseline + 0.5 * (y_peak - baseline)
                l = int(i)
                while l > 0 and y_s[l] > half:
                    l -= 1
                r = int(i)
                while r < len(y_s) - 1 and y_s[r] > half:
                    r += 1
                fwhm.append(float(x[r] - x[l]) if r > l else 0.0)

    # ---- filter out amorphous hump peaks (too broad) ----
    fwhm_max_deg = 3.0
    if len(peaks) and fwhm:
        fwhm_arr = np.array(fwhm, dtype=float)
        keep = fwhm_arr <= fwhm_max_deg
        peaks = peaks[keep]
        prominences = prominences[keep] if len(prominences) else prominences
        fwhm = fwhm_arr[keep].tolist()

    peak_pos = x[peaks].tolist() if len(peaks) else []
    peak_abs = y_s[peaks].tolist() if len(peaks) else []
    peak_height = (y_s[peaks] - baseline).tolist() if len(peaks) else []

    peak_prom = prominences.tolist() if len(prominences) else []
    fwhm_list = list(fwhm) if fwhm else []

    median_fwhm = float(np.median(fwhm_list)) if fwhm_list else None
    mean_fwhm = float(np.mean(fwhm_list)) if fwhm_list else None

    # ---- relative prominence (to suppress low-angle background ripples) ----
    rel_prom = [float(pr) / (float(pa) + 1e-12) for pr, pa in zip(peak_prom, peak_abs)] if peak_prom else []
    LOW_REL_PROM_MIN = 0.06  # heuristic; raise if still false-D; lower if missing true low-angle peaks

    # IMPORTANT: redefine low_angle_peak_count_lt10 to be robust
    low_mask = [(p >= 2.0 and p < 10.0 and rp >= LOW_REL_PROM_MIN) for p, rp in zip(peak_pos, rel_prom)]
    low_angle_peak_count = int(sum(low_mask))
    low_angle_rel_prom_max = float(max([rp for p, rp in zip(peak_pos, rel_prom) if 2.0 <= p < 10.0] or [0.0]))
    low_angle_peak_positions_2_10 = [p for p, m in zip(peak_pos, low_mask) if m][:30]

    # high-angle peaks (small-molecule / salt / raw material signature)
    high_angle_peak_count_gt15 = int(sum(p > 15.0 for p in peak_pos))
    high_angle_peak_positions_gt15 = [p for p in peak_pos if p > 15.0][:30]

    # ---- areas (amorphous index) ----
    pos_area = float(np.trapz(np.clip(resid, 0, None), x))
    hump_mask = (x >= 15.0) & (x <= 30.0)
    hump_area = float(np.trapz(np.clip(resid[hump_mask], 0, None), x[hump_mask])) if np.any(hump_mask) else None
    hump_ratio = float(hump_area / pos_area) if (hump_area is not None and pos_area > 0) else None

    mid_peaks = sum(1 for p in peak_pos if 10.0 <= p <= 30.0)
    sharp_peaks = sum(1 for w in fwhm_list if w < 0.5) if fwhm_list else 0

    # ---- SNR: peak-based with a floor tied to y_ref (prevents absurdly huge SNR) ----
    noise_snr = max(noise_peak, 0.005 * y_ref)
    snr_peak_max = float((max(peak_prom) / (noise_snr + 1e-12)) if peak_prom else 0.0)

    cryst_score = float(len(peaks) * snr_peak_max / (median_fwhm + 1e-6)) if median_fwhm else float(len(peaks) * snr_peak_max)

    return {
        "n_points": int(len(x)),
        "two_theta_min": float(x.min()),
        "two_theta_max": float(x.max()),

        "baseline_p10": baseline,
        "noise_est": float(noise_peak),         # detection noise
        "noise_snr_floor": float(noise_snr),    # for interpretability/debug

        "snr_max": snr_peak_max,
        "peak_count": int(len(peaks)),

        # NOTE: this key is used by your current prompt; now it is robust (2–10° + rel-prom filter)
        "low_angle_peak_count_lt10": int(low_angle_peak_count),
        "mid_angle_peak_count_10_30": int(mid_peaks),
        "high_angle_peak_count_gt15": int(high_angle_peak_count_gt15),

        "peak_positions": peak_pos[:30],
        "peak_heights": peak_height[:30],
        "peak_prominences": peak_prom[:30],
        "peak_fwhm_deg": fwhm_list[:30],
        "median_fwhm_deg": median_fwhm,
        "mean_fwhm_deg": mean_fwhm,

        "pos_integrated_intensity": pos_area,
        "hump_area_15_30": hump_area,
        "hump_ratio_15_30": hump_ratio,

        "sharp_peaks_count": int(sharp_peaks),
        "crystallinity_score_proxy": cryst_score,


        "low_angle_rel_prom_max": low_angle_rel_prom_max,
        "low_angle_peak_positions_2_10": low_angle_peak_positions_2_10,
        "high_angle_peak_positions_gt15": high_angle_peak_positions_gt15,

        "note": "low_angle_peak_count_lt10 uses 2–10° & rel_prom>=0.06; y_ref uses 5–30°; snr uses noise floor 0.005*y_ref",
    }

async def agent3_plan_round(
    llm: GoogleGenAI,
    ligands: List[str],
    agent2_out: Dict[str, Any],
    round_idx: int,
    settings: Settings,
) -> Dict[str, Any]:
    prior = agent2_out.get("final_prior", {}) or {}
    prov = (agent2_out.get("provenance", {}) or {}) if settings.agent3_pass_provenance else {}
    rationale = agent2_out.get("decider_rationale") or ""


    if settings.plan_mode in ("llm", "llm_then_deterministic"):
        prompt = safe_format_prompt(
            PROMPT_AGENT3_PLAN,
            round_idx=round_idx,
            decider_rationale=rationale,
            ligands_json=json.dumps(ligands, ensure_ascii=False),
            final_prior_json=json.dumps(prior, ensure_ascii=False),
            provenance_json=json.dumps(prov, ensure_ascii=False),
        )
        print("\n")
        print("prompt3plan\n")
        print(prompt)
        try:
            js = await llm_complete_json(llm, prompt, max_retries=3)
            if isinstance(js, dict) and isinstance(js.get("experiments", None), list) and len(js["experiments"]) >= 1:
                return js
            raise ValueError("Agent3 plan JSON invalid (missing experiments list).")
        except Exception:
            if settings.plan_mode == "llm":
                raise
            return deterministic_systematic_plan(prior, round_idx=round_idx, n_exps=settings.round_experiments)

    return deterministic_systematic_plan(prior, round_idx=round_idx, n_exps=settings.round_experiments)

def compress_pxrd_features_for_llm(pxrd_feat: Any) -> Any:

    if not isinstance(pxrd_feat, dict):
        return pxrd_feat
    keep = [
        "low_angle_peak_count_lt10",
        "hump_ratio_15_30",
        "crystallinity_score_proxy",
        "snr_max",
        "peak_count",
        "median_fwhm_deg",
        "low_angle_peak_positions_2_10",
        "high_angle_peak_positions_gt15",
        "error",
        "file",
        "note",
    ]
    return {k: pxrd_feat.get(k) for k in keep if k in pxrd_feat}

async def agent3_analyze_batch(llm: GoogleGenAI, exp_packages: List[Dict[str, Any]]) -> Dict[str, Any]:
    prompt = safe_format_prompt(
        PROMPT_AGENT3_BATCH_ANALYZE,
        taxonomy=FAILURE_TAXONOMY_TEXT,
        batch_json=json.dumps(exp_packages, ensure_ascii=False),
    )
    print("\n")
    print("prompt3classify\n")
    print(prompt)
    # if settings.debug_print_prompts: print(prompt)
    return await llm_complete_json(llm, prompt, max_retries=3)

async def agent3_classify_one(llm: GoogleGenAI, exp_package: Dict[str, Any]) -> Dict[str, Any]:
    prompt = safe_format_prompt(
        PROMPT_AGENT3_CLASSIFY,
        taxonomy=FAILURE_TAXONOMY_TEXT,
        exp_package_json=json.dumps(exp_package, ensure_ascii=False),
    )
    print("\n")
    print("prompt3classify\n")
    print(prompt)
    return await llm_complete_json(llm, prompt, max_retries=3)
async def agent3_batch_classify_vision(
    llm: GoogleGenAI,
    exp_items: List[Dict[str, Any]],
    *,
    fewshot_paths: Optional[Sequence[str]] = None,
) -> Dict[str, Any]:
    header = safe_format_prompt(
        PROMPT_AGENT3_BATCH_VISION,
        taxonomy=FAILURE_TAXONOMY_TEXT,
    )

    blocks = [TextBlock(text=header)]

    # --- FEW-SHOT:
    if fewshot_paths:
        blocks.append(TextBlock(
            text=(
                "FEW-SHOT REFERENCE (CONFIRMED LABELS):\n"
                "The next TWO PXRD images are confirmed Class D (Near-crystalline / poorly crystalline).\n"
                "Use them to calibrate what counts as '>=2 real broadened peaks above noise'.\n"
            )
        ))
        for i, p in enumerate(fewshot_paths, start=1):
            if not p or not isinstance(p, str) or not os.path.exists(p):
                raise FileNotFoundError(f"Few-shot Class D image not found: {p}")
            blocks.append(TextBlock(text=f"FEWSHOT_EXAMPLE_{i}: CONFIRMED label = Class D"))
            blocks.append(ImageBlock(path=p, image_mimetype=guess_image_mime(p)))

        blocks.append(TextBlock(
            text="--- END OF FEW-SHOT. Now classify the following experiments. ---"
        ))

    for item in exp_items:
        meta = {
            "exp_id": item.get("exp_id"),
            "planned": item.get("planned"),
            "observations": item.get("observations", ""),
        }
        blocks.append(TextBlock(text="EXPERIMENT_JSON:\n" + json.dumps(meta, ensure_ascii=False)))

        img_path = item.get("pxrd_image_path")
        if img_path:
            mime = item.get("pxrd_image_mime") or guess_image_mime(img_path)
            blocks.append(ImageBlock(path=img_path, image_mimetype=mime))

    msg = ChatMessage(role="user", blocks=blocks)
    return await llm_chat_json(llm, [msg], max_retries=3)


async def agent3_update_posterior_and_plan(
    llm: GoogleGenAI,
    current_prior: Dict[str, Any],
    history: Dict[str, Any],
) -> Dict[str, Any]:
    prompt = safe_format_prompt(
        PROMPT_AGENT3_UPDATE,
        current_prior_json=json.dumps(current_prior, ensure_ascii=False),
        history_json=json.dumps(history, ensure_ascii=False),
    )
    # print("\n")
    # print("prompt3 update\n")
    # print(prompt)
    return await llm_complete_json(llm, prompt, max_retries=3)


# =============================================================================
# 8) SESSION PERSISTENCE (reviewer-friendly)
# =============================================================================

def _json_safe(obj: Any) -> Any:
    """
    Make obj JSON-serializable:
    - set -> sorted list
    - numpy types -> python scalars
    - numpy arrays -> lists
    """
    if isinstance(obj, set):
        return sorted(list(obj))
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

def _write_json(path: str, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_json_safe)

def _append_jsonl(path: str, obj: Any) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=_json_safe) + "\n")

async def persist_session(ctx: Context, settings: Settings) -> None:
    mode = await ctx.store.get("mode", default="idle")
    state = await ctx.store.get("state", default={})
    payload = {"saved_at": _now_iso(), "mode": mode, "state": state}
    _write_json(settings.session_json_path, payload)

async def restore_session_if_needed(ctx: Context, settings: Settings) -> None:
    loaded = await ctx.store.get("__session_loaded", default=False)
    if loaded:
        return

    if os.path.exists(settings.session_json_path):
        try:
            with open(settings.session_json_path, "r", encoding="utf-8") as f:
                payload = json.load(f)
            await ctx.store.set("mode", payload.get("mode", "idle"))
            await ctx.store.set("state", payload.get("state", {}))
        except Exception:
            await ctx.store.set("mode", "idle")
            await ctx.store.set("state", {})

    await ctx.store.set("__session_loaded", True)

def audit_log(settings: Settings, record: Dict[str, Any]) -> None:
    if not settings.audit_jsonl_path:
        return
    record = {"ts": _now_iso(), **record}
    _append_jsonl(settings.audit_jsonl_path, record)


# =============================================================================
# 9) WORKFLOW EVENTS
# =============================================================================

class SynthesisQueryEvent(Event):
    user_msg: str

class FeedbackEvent(Event):
    user_msg: str


# =============================================================================
# 10) WORKFLOW (automatic routing plan/feedback + adaptive topk + persistence)
# =============================================================================

class COFIntegratedWorkflow(Workflow):
    def __init__(self, settings: Settings, **kwargs):
        super().__init__(**kwargs)
        self.s = settings

        # init LLM
        api_key = os.getenv(settings.google_api_key_env, "")
        if not api_key:
            raise EnvironmentError(f"Missing env {settings.google_api_key_env}")
        self.llm = GoogleGenAI(
            model=settings.llm_model,
            api_key=api_key,
            temperature=settings.llm_temperature,
        )

        # init DB + embedder + retriever
        self.db = RAGDatabaseV2(settings)
        self.db.load()
        self.embedder = DashScopeEmbedder(settings)
        self.retriever = SetRetrieverV2(self.db, self.embedder)

    # -------- Router --------
    @step
    async def route(self, ctx: Context, ev: StartEvent) -> Union[SynthesisQueryEvent, FeedbackEvent, StopEvent]:
        await restore_session_if_needed(ctx, self.s)

        user_msg = ev.user_msg if hasattr(ev, "user_msg") else str(ev)

        # reset
        if user_msg.strip().lower() in {"reset", "/reset", "new", "/new"}:
            await ctx.store.set("mode", "idle")
            await ctx.store.set("state", {})
            await persist_session(ctx, self.s)
            audit_log(self.s, {"event": "reset"})
            return StopEvent(result={"mode": "idle", "message": "Workflow reset. State cleared."})

        mode = await ctx.store.get("mode", default="idle")
        if mode == "awaiting_feedback":
            return FeedbackEvent(user_msg=user_msg)
        return SynthesisQueryEvent(user_msg=user_msg)

    # -------- Adaptive retrieval --------
    async def _retrieve_adaptive(self, ligands: List[str]) -> Dict[str, Any]:
        """
        Adaptive top-k growth:
        expand top_k until evidence diversity meets thresholds or attempts exhausted.
        """
        k = int(self.s.topk_init)
        attempts = 0
        last_pack: Dict[str, Any] = {"indices": [], "scores": [], "hits": [], "records": []}

        while True:
            attempts += 1
            pack = self.retriever.search(ligands, top_k=k)
            last_pack = pack

            indices = pack.get("indices", [])
            if not indices:
                break

            ev = self.db.evidence_sets_by_indices(indices)
            ok = (len(ev.get("solvents", [])) >= self.s.min_evid_solvent_unique) and (len(ev.get("catalysts", [])) >= self.s.min_evid_catalyst_unique)
            if ok:
                break

            if attempts >= self.s.topk_attempts_max or k >= self.s.topk_max:
                break

            k = min(int(k * self.s.topk_grow), int(self.s.topk_max))

        last_pack["top_k_used"] = int(k)
        last_pack["top_k_attempts"] = int(attempts)
        last_pack["evidence_sets"] = self.db.evidence_sets_by_indices(last_pack.get("indices", []))
        return last_pack

    # -------- PLAN: Agent1 -> Agent2 -> Agent3 --------
    @step
    async def handle_query(self, ctx: Context, ev: SynthesisQueryEvent) -> StopEvent:
        user_msg = ev.user_msg

        # --- Agent1: extract ligands (2 or 3) ---
        ligq = await agent1_extract_ligands(self.llm, user_msg)
        ligands = ligq.ligands

        # --- Agent1: retrieval ---
        retrieval = await self._retrieve_adaptive(ligands)
        hits = retrieval.get("hits", [])
        topk_summary = self.db.topk_summary_string(hits, k=min(50, len(hits)))

        # --- exact-match shortcut ---
        exact = agent1_find_exact_match_multi(ligands, retrieval.get("records", []))
        if exact is not None:
            direct = format_direct_answer_from_record(exact)
            await ctx.store.set("mode", "idle")
            await ctx.store.set("state", {})
            await persist_session(ctx, self.s)

            audit_log(self.s, {
                "event": "direct_match",
                "ligands": ligands,
                "top_k_used": retrieval.get("top_k_used"),
                "topk_summary": topk_summary,
                "direct_answer": direct,
            })
            return StopEvent(result=direct)

        # --- Agent2: robust prior ---
        agent2_out = await agent2_generate_prior(
            llm=self.llm,
            db=self.db,
            ligands=ligands,
            retrieval_pack=retrieval,
            settings=self.s,
        )

        # --- Agent3: round-1 plan ---
        round_idx = 1
        plan = await agent3_plan_round(
            llm=self.llm,
            ligands=ligands,
            agent2_out=agent2_out,
            round_idx=round_idx,
            settings=self.s,
        )

        # store state
        state = {
            "created_at": _now_iso(),
            "ligands": ligands,
            "retrieval": {
                "bucket": retrieval.get("bucket"),
                "query_ligands": retrieval.get("query_ligands"),
                "top_k_used": retrieval.get("top_k_used"),
                "top_k_attempts": retrieval.get("top_k_attempts"),
                "indices": retrieval.get("indices"),
                "scores": retrieval.get("scores"),
                "evidence_sets": retrieval.get("evidence_sets"),
                "topk_summary_string": topk_summary,
            },
            "agent2": agent2_out,
            "history": {"rounds": [{"round": round_idx, "plan": plan, "feedback": None, "classifications": None}]},
        }
        await ctx.store.set("state", state)
        await ctx.store.set("mode", "awaiting_feedback")
        await persist_session(ctx, self.s)

        audit_log(self.s, {
            "event": "plan_round1",
            "ligands": ligands,
            "bucket": retrieval.get("bucket"),
            "top_k_used": retrieval.get("top_k_used"),
            "top_k_attempts": retrieval.get("top_k_attempts"),
            "topk_summary_string": topk_summary,
            "agent2_selected_strategy": agent2_out.get("selected_strategy"),
            "agent2_merge_mode_used": agent2_out.get("merge_mode_used"),
            "final_prior": agent2_out.get("final_prior"),
            "plan": plan,
        })

        out = {
            "mode": "awaiting_feedback",
            "message": "Round-1 plan generated. Execute any subset, then send feedback JSON in the next message.",
            "ligands": ligands,
            "retrieval": {
                "bucket": retrieval.get("bucket"),
                "top_k_used": retrieval.get("top_k_used"),
                "top_k_attempts": retrieval.get("top_k_attempts"),
                "evidence_sets": retrieval.get("evidence_sets"),
                "topk_summary_string": topk_summary,
            },
            "agent2": {
                "selected_strategy": agent2_out.get("selected_strategy"),
                "merge_mode_used": agent2_out.get("merge_mode_used"),
                "strategy_reports": agent2_out.get("strategy_reports"),
                "final_prior": agent2_out.get("final_prior"),
                "deterministic_prior": agent2_out.get("deterministic_prior"),
                "llm_decider": agent2_out.get("llm_decider"),
                "provenance": agent2_out.get("provenance"),
            },
            "plan": plan,
            "feedback_schema_hint": {
                "executed_experiments": [
                    {
                        "exp_id": "R1-E01",
                        "executed": True,
                        "observations": "precipitate immediately / gel / clear / oil / color change ...",
                        "pxrd_image_path": "path/to/pxrd.png",
                        "pxrd_image_mime": "image/png"
                    }
                ]
            }
        }
        return StopEvent(result=out)

    # -------- FEEDBACK loop --------
    @step
    async def handle_feedback(self, ctx: Context, ev: FeedbackEvent) -> StopEvent:
        user_msg = ev.user_msg

        state = await ctx.store.get("state", default={})
        if not state:
            await ctx.store.set("mode", "idle")
            await persist_session(ctx, self.s)
            return StopEvent(result={"mode": "idle", "error": "No active state. Start with a new synthesis query."})

        ligands = state.get("ligands", [])
        agent2_out = state.get("agent2", {})
        history = state.get("history", {"rounds": []})

        # parse feedback JSON
        try:
            feedback_js = json.loads(extract_json_object(user_msg))
        except Exception:
            repair_prompt = f"""
Convert the following user feedback into JSON strictly with this schema:
{{
  "executed_experiments": [
    {{
      "exp_id": "R1-E01",
      "executed": true,
      "observations": "...",
      "pxrd_image_path": "...",
      "pxrd_image_mime": "image/png"
    }}
  ]
}}
User feedback:
{user_msg}
Return JSON only.
"""

            feedback_js = await llm_complete_json(self.llm, repair_prompt, max_retries=3)

        executed = feedback_js.get("executed_experiments", []) or []

        last_round = history["rounds"][-1] if history.get("rounds") else {}
        last_plan = (last_round.get("plan", {}) or {}).get("experiments", []) or []
        plan_map = {p.get("exp_id"): p for p in last_plan if isinstance(p, dict) and p.get("exp_id")}
        

        batch_items = []
        for ex in executed:
            if not ex.get("executed", True):
                continue
        
            exp_id = ex.get("exp_id")
            img_path = ex.get("pxrd_image_path") 
            img_mime = ex.get("pxrd_image_mime") 
        
            if img_path and (not isinstance(img_path, str) or not os.path.exists(img_path)):
                img_path = None
        
            batch_items.append({
                "exp_id": exp_id,
                "planned": plan_map.get(exp_id),
                "observations": ex.get("observations", ""),
                "pxrd_image_path": img_path,
                "pxrd_image_mime": img_mime,
            })
        
        if not batch_items:
            return StopEvent(result={
                "mode": "awaiting_feedback",
                "message": "No executed experiments found in feedback. Please include executed_experiments with executed=true.",
                "last_plan": last_plan,
            })
        
        fewshot_paths = self.s.fewshot_classD_image_paths if self.s.fewshot_enable else None
        batch_out = await agent3_batch_classify_vision(
            self.llm,
            batch_items,
            fewshot_paths=fewshot_paths,
        )

        
        cls_list = batch_out.get("experiments", []) or []
        cls_map = {
            c.get("exp_id"): c
            for c in cls_list
            if isinstance(c, dict) and c.get("exp_id")
        }
        round_summary = batch_out.get("round_summary", "")
        
        classifications = cls_list
        
        enriched_exec = []
        for item in batch_items:
            exp_id = item.get("exp_id")
            enriched_exec.append({
                "input": item,
                "analysis": cls_map.get(exp_id, None),
            })
        
        if history.get("rounds"):
            history["rounds"][-1]["feedback"] = feedback_js
            history["rounds"][-1]["classifications"] = classifications
            history["rounds"][-1]["executed_enriched"] = enriched_exec
            history["rounds"][-1]["round_summary"] = round_summary
        
        if any((c.get("label") == "SUCCESS") for c in classifications if isinstance(c, dict)):
            await ctx.store.set("mode", "completed")
            state["history"] = history
            await ctx.store.set("state", state)
            await persist_session(ctx, self.s)
            audit_log(self.s, {"event": "completed_success", "ligands": ligands, "classifications": classifications})
            return StopEvent(result={
                "mode": "completed",
                "message": "SUCCESS detected. Workflow terminated.",
                "classifications": classifications,
                "history": history,
            })
        
        executed_compact = []
        for item in enriched_exec:
            ex_in = item.get("input") or {}
            exp_id = ex_in.get("exp_id")
            ana = item.get("analysis") or {}
        
            executed_compact.append({
                "exp_id": exp_id,
                "planned": plan_map.get(exp_id),
                "observations": ex_in.get("observations", ""),
                "analysis": {
                    "label": ana.get("label"),
                    "confidence": ana.get("confidence"),
                    "rationale": ana.get("rationale"),
                    "tactical_hints": ana.get("tactical_hints"),
                }
            })
        
        round_compact = {
            "round": last_round.get("round"),
            "plan_experiments": last_plan,
            "executed": executed_compact,
            "round_summary": round_summary,
        }



        current_prior = agent2_out.get("final_prior", {}) or {}
        upd = await agent3_update_posterior_and_plan(
            self.llm,
            current_prior=current_prior,
            history=round_compact,  
        )


        new_round_idx = len(history.get("rounds", [])) + 1
        if new_round_idx > self.s.max_rounds:
            await ctx.store.set("mode", "completed")
            state["history"] = history
            await ctx.store.set("state", state)
            await persist_session(ctx, self.s)
            return StopEvent(result={
                "mode": "completed",
                "message": f"Max rounds reached ({self.s.max_rounds}). Stopping.",
                "last_classifications": classifications,
                "history": history,
            })

        if isinstance(upd, dict) and "updated_prior" in upd:
            try:
                _ = PriorCard(**upd["updated_prior"])
                agent2_out["final_prior"] = upd["updated_prior"]
            except ValidationError:
                pass

        next_plan = upd.get("next_round_plan", None) if isinstance(upd, dict) else None
        if next_plan is None:
            next_plan = await agent3_plan_round(
                llm=self.llm,
                ligands=ligands,
                agent2_out=agent2_out,
                round_idx=new_round_idx,
                settings=self.s,
            )

        history["rounds"].append({"round": new_round_idx, "plan": next_plan, "feedback": None, "classifications": None})

        state["agent2"] = agent2_out
        state["history"] = history
        await ctx.store.set("state", state)
        await ctx.store.set("mode", "awaiting_feedback")
        await persist_session(ctx, self.s)

        audit_log(self.s, {
            "event": "plan_next_round",
            "ligands": ligands,
            "round": new_round_idx,
            "diagnosis_summary": upd.get("diagnosis_summary") if isinstance(upd, dict) else None,
            "updated_prior": agent2_out.get("final_prior"),
            "next_plan": next_plan,
            "last_classifications": classifications,
        })

        return StopEvent(result={
            "mode": "awaiting_feedback",
            "message": "Next round plan generated. Execute any subset and send feedback JSON again.",
            "diagnosis_summary": upd.get("diagnosis_summary") if isinstance(upd, dict) else None,
            "updated_prior": agent2_out.get("final_prior", None),
            "next_plan": next_plan,
            "last_classifications": classifications,
        })


# =============================================================================
# 11) Helper
# =============================================================================

def pretty_print(obj: Any) -> None:
    print(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_safe))


In [19]:
os.environ["DASHSCOPE_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
settings = Settings(
    rag_db_path="rag_database_v2",
    records_file="rag_database/cof_data.json", 
)
wf = COFIntegratedWorkflow(settings=settings, timeout=3600, verbose=False)
ctx = Context(wf)

res = await wf.run(ctx=ctx, user_msg="reset")
pretty_print(res)

[DBv2] records=2709 | set2=2189 | set3=284 | faiss=True
{
  "mode": "idle",
  "message": "Workflow reset. State cleared."
}


In [20]:
os.environ["DASHSCOPE_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
settings = Settings(
    rag_db_path="rag_database_v2",
    records_file="rag_database/cof_data.json",
    repeats_per_strategy = 5,
    strategies= ("C","D")
)
wf = COFIntegratedWorkflow(settings=settings, timeout=3600, verbose=False)
ctx = Context(wf)

res = await wf.run(ctx=ctx, user_msg="1,3,6,8-Tetrakis(4-aminophenyl)pyrene (TAPPy) + 2',3',5',6'-Tetrafluorobiphenyl-4,4'-dicarbaldehyde")
pretty_print(res)

[DBv2] records=2709 | set2=2189 | set3=284 | faiss=True


prior prompt


You are an expert-level researcher specializing in the methodology of Covalent Organic Framework (COF) synthesis. Your core task is to provide a SCIENTIFIC, recommendation for the experimental condition range for the synthesis of a new, unknown COF, based on a series of relevant COF synthesis cases retrieved from a database. Derive synthesis patterns from analogous cases in the database, and subsequently formulate a rational parameter space (solvent, catalyst, temperature, time) specifically tailored to this novel ligand combination.
# Target for Synthesis
This new COF will be constructed from the following ligands:
["1,3,6,8-Tetrakis(4-aminophenyl)pyrene (TAPPy)", "2',3',5',6'-Tetrafluorobiphenyl-4,4'-dicarbaldehyde"]
# Instructions and Framework for Analysis
Please strictly follow the steps below to meticulously analyze the provided cases and summarize the recommended condition range. 
1. Solvent System Analysis

In [21]:
plan = res["plan"]["experiments"]
for e in plan:
    print(e["exp_id"], e["solvent_system"], e["solvent_detail"], e["catalyst"],e["catalyst_detail"], e["temperature_C"], e["time_h"])


R1-E01 o-dichlorobenzene/n-butanol 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E02 1,4-dioxane/mesitylene 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E03 mesitylene/benzyl alcohol 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E04 o-dichlorobenzene/n-butanol 3:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E05 o-dichlorobenzene/n-butanol 1:3 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E06 o-dichlorobenzene/n-butanol 1:1 v/v Aqueous Acetic Acid 0.1 mL 3.0 M Aqueous Acetic Acid 120 72
R1-E07 o-dichlorobenzene/n-butanol 1:1 v/v Aqueous Acetic Acid 0.1 mL 9.0 M Aqueous Acetic Acid 120 72
R1-E08 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R1-E09 1,4-dioxane/mesitylene 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R1-E10 1,4-dioxane/mesitylene 1:1 v/v Aqueous Acetic Acid 0.1 mL 3.0 M Aqueous Acetic Acid 120 72


In [14]:
os.environ["DASHSCOPE_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
settings = Settings(
    rag_db_path="rag_database_v2",
    records_file="rag_database/cof_data.json",  
)
wf = COFIntegratedWorkflow(settings=settings, timeout=3600, verbose=False)
ctx = Context(wf)

res = await wf.run(ctx=ctx, user_msg="reset")
pretty_print(res)

[DBv2] records=2709 | set2=2189 | set3=284 | faiss=True
{
  "mode": "idle",
  "message": "Workflow reset. State cleared."
}


In [15]:
os.environ["DASHSCOPE_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"] = ""
settings = Settings(
    rag_db_path="rag_database_v2",
    records_file="rag_database/cof_data.json",
    repeats_per_strategy = 5,
    strategies= ("C","D")
)
wf = COFIntegratedWorkflow(settings=settings, timeout=3600, verbose=False)
ctx = Context(wf)

res = await wf.run(ctx=ctx, user_msg="1,3,6,8-Tetrakis(4-aminophenyl)pyrene (TAPPy) + 2,2',3,3',5,5',6,6'-Octafluoro-4,4'-biphenyldicarboxaldehyde")
pretty_print(res)

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x000002131DDD5F10>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000021324214CA0>


[DBv2] records=2709 | set2=2189 | set3=284 | faiss=True


prior prompt


You are an expert-level researcher specializing in the methodology of Covalent Organic Framework (COF) synthesis. Your core task is to provide a SCIENTIFIC, recommendation for the experimental condition range for the synthesis of a new, unknown COF, based on a series of relevant COF synthesis cases retrieved from a database. Derive synthesis patterns from analogous cases in the database, and subsequently formulate a rational parameter space (solvent, catalyst, temperature, time) specifically tailored to this novel ligand combination.
# Target for Synthesis
This new COF will be constructed from the following ligands:
["1,3,6,8-Tetrakis(4-aminophenyl)pyrene (TAPPy)", "2,2',3,3',5,5',6,6'-Octafluoro-4,4'-biphenyldicarboxaldehyde"]
# Instructions and Framework for Analysis
Please strictly follow the steps below to meticulously analyze the provided cases and summarize the recommended condition range. 
1. Solvent System

In [16]:
plan = res["plan"]["experiments"]
for e in plan:
    print(e["exp_id"], e["solvent_system"], e["solvent_detail"], e["catalyst"],e["catalyst_detail"], e["temperature_C"], e["time_h"])


R1-E01 mesitylene/1,4-dioxane 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E02 mesitylene/1,4-dioxane 1:1 v/v Aqueous Acetic Acid 0.1 mL 3.0 M Aqueous Acetic Acid 120 72
R1-E03 mesitylene/1,4-dioxane 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R1-E04 o-dichlorobenzene/n-butanol 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E05 o-dichlorobenzene/n-butanol 1:1 v/v Aqueous Acetic Acid 0.1 mL 3.0 M Aqueous Acetic Acid 120 72
R1-E06 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R1-E07 anisole/1,4-dioxane 1:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E08 anisole/1,4-dioxane 1:1 v/v Aqueous Acetic Acid 0.1 mL 3.0 M Aqueous Acetic Acid 120 72
R1-E09 mesitylene/1,4-dioxane 2:1 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72
R1-E10 mesitylene/1,4-dioxane 1:2 v/v Aqueous Acetic Acid 0.1 mL 6.0 M Aqueous Acetic Acid 120 72


In [17]:
import json

feedback = {
  "executed_experiments": [
    {
      "exp_id": "R1-E01",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd1.png",
      "pxrd_image_mime": "image/png"
    },
    {
      "exp_id": "R1-E02",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd2.png"
    },
      {
      "exp_id": "R1-E03",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd3.png"
    },
      {
      "exp_id": "R1-E04",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd4.png"
    },
      {
      "exp_id": "R1-E05",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd5.png"
    },
    {
      "exp_id": "R1-E06",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd6.png"
    },
    {
      "exp_id": "R1-E07",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd7.png"
    },
        {
      "exp_id": "R1-E08",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd8.png"
    },
    {
      "exp_id": "R1-E9",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd9.png"
    },
      {
      "exp_id": "R1-E10",
      "executed": True,
      "observations": "Powdery solid",
      "pxrd_image_path": r"pxrd/pxrd10.png"
    },
  ]
}

res2 = await wf.run(ctx=ctx, user_msg=json.dumps(feedback, ensure_ascii=False))
pretty_print(res2)


{
  "mode": "awaiting_feedback",
  "message": "Next round plan generated. Execute any subset and send feedback JSON again.",
  "diagnosis_summary": "The initial screening round revealed a widespread failure mode of Class C (Amorphous Kinetic Trap), indicating that most conditions promoted rapid, disordered polymerization. The critical breakthrough was experiment R1-E06, which yielded a Class D (Near-Crystalline) material using an o-dichlorobenzene/n-butanol solvent system with Trifluoroacetic Acid (TFA) as the catalyst. This result strongly suggests that the combination of this specific solvent environment and a strong acid catalyst is essential for achieving the necessary reaction reversibility for crystallization. The weaker acetic acid catalyst was insufficient across all solvent systems. Therefore, the strategy for Round 2 is to pivot entirely to the promising oDCB/nBuOH/TFA system and perform a focused optimization. We will fine-tune the key parameters—catalyst loading, solvent ra

In [18]:
plan = res2["next_plan"]
for e in plan:
    print(e["exp_id"], e["solvent_system"], e["solvent_detail"], e["catalyst"],e["catalyst_detail"], e["temperature_C"], e["time_h"])


R2-E01 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R2-E02 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.05 mL 1.0 M TFA 120 72
R2-E03 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.2 mL 1.0 M TFA 120 72
R2-E04 o-dichlorobenzene/n-butanol 2:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R2-E05 o-dichlorobenzene/n-butanol 1:2 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 72
R2-E06 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 130 72
R2-E07 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 110 72
R2-E08 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 120 120
R2-E09 o-dichlorobenzene/n-butanol 2:1 v/v Trifluoroacetic Acid 0.2 mL 1.0 M TFA 120 72
R2-E10 o-dichlorobenzene/n-butanol 1:1 v/v Trifluoroacetic Acid 0.1 mL 1.0 M TFA 130 120
